# 🏦 Consolidación de Datos de Transacciones

**Proyecto:** Pipeline de Datos Financieros - Prueba Técnica Iris CF  
**Ingeniero:** Nelson Andrés Salinas Zapata  
**Tecnología:** PySpark (Local), Pandas. 

----

## ⚙️ Paso 0: Inicialización del Entorno y Carga Cruda
En esta sección encendemos el motor de PySpark configurando una sesión local que aproveche todos los hilos del procesador (`local[*]`). Cargaremos las fuentes en su estado puramente crudo para diagnosticar cómo vienen estructuradas desde los sistemas de origen.

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# --- CONFIGURACIÓN DE FORMATO DE IMPRESIÓN ---
SEPARATOR = "=" * 90
SUB_LINE = "-" * 90

# 1. Crear la sesión de Spark optimizada para desarrollo local
# Usamos un formato de fechas ideal para fechas actuales
spark = SparkSession.builder \
    .appName("IrisFinancialExploration") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .getOrCreate()

# Ajustar el nivel de log a WARN para mantener la consola limpia
spark.sparkContext.setLogLevel("WARN")

print("✨ ¡Motor PySpark encendido correctamente en local! ✨")

# 2. Definición de rutas a las fuentes crudas
PATH_TX = "../data/raw/transacciones.csv"
PATH_CLI = "../data/raw/clientes.json"

# 3. Lectura inicial descriptiva
df_transacciones_raw = spark.read.csv(PATH_TX, sep =";", header=True)
df_clientes_raw = spark.read.option("multiLine", True).json(PATH_CLI)

print("\n📊 CARGA EXITOSA: Fuentes cargadas en memoria para diagnóstico.")

✨ ¡Motor PySpark encendido correctamente en local! ✨

📊 CARGA EXITOSA: Fuentes cargadas en memoria para diagnóstico.


---

## 🔍 Paso 1: Exploración

Antes de transformar los datos se realizará un exploratorio exhaustivo de las dos fuentes de datos proporcionadas. Se realizará una inspección y muestreo inicial, para luego realizar un diagnóstico para cada fuente.

### 🕵️‍♂️ Inspección de Esquemas y Muestreo Inicial

In [2]:


print(SEPARATOR)
print("🔍 ESTRUCTURA CRUDA DE CLIENTES")
print(SEPARATOR)
df_clientes_raw.printSchema()
df_clientes_raw.show(20, truncate=False)

print(SEPARATOR)
print("🔍 ESTRUCTURA CRUDA DE TRANSACCIONES")
print(SEPARATOR)

df_transacciones_raw.printSchema()
df_transacciones_raw.show(20, truncate=False)

🔍 ESTRUCTURA CRUDA DE CLIENTES
root
 |-- activo: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- contacto: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- telefono: string (nullable = true)
 |-- fecha_alta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- productos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- segmento: string (nullable = true)
 |-- tipo_documento: string (nullable = true)

+------+------------+------------------------------------------------+-----------+----------+------------------+----------------+---------------------------------------+--------+--------------+
|activo|ciudad      |contacto                                        |fecha_alta |id_cliente|nombre            |numero_documento|productos                              |segmento|tipo_documento|
+------+------------+-------

### 📝 Hallazgos (Inspección de Esquemas y Muestreo Inicial)
Viendo las primeras 20 filas de cada tabla se identifican unos datos bastante sucios, por lo que se procederá a hacer un análisis exaustivo por columnas. Como elementos a resaltar identificamos:

*   **Clientes (`.json`):**
    *   *Formato JSON:* El archivo viene estructurado en formato tradicional (*"multilinea"*) y no en el formato por defecto de Spark (*"JSON Lines"*). Además se identifican datos anidados en los atributos `contacto` y `productos`.
    *   *Estado de las columnas:* En todas, menos en `id_cliente` (aparentemente) se identifica un problema de no estandarización de formato y de valores.
    *   *Posibles duplicados*: Se observan indicios de posibles filas duplicadas, pues se ven nombres repetidos y números de documento nulos. Es necesario asegurar que los valores de `id_cliente` y de `numero_documento` sean únicos.
*   **Transacciones (`.csv`):** 
    *   *Separador detectado:* Haciendo una inspección rápida del csv antes de cargar los datos se identificó que se usa `;` como separador.
    *   *Estado de las columnas:* Similar a como pasa en las transacciones, se observa un problema de no estandarización de formato y valores en todas las columnas menos `id_transaccion` e `id_cliente`.
    *   *Control de valores:* Es necesario comprobar que el `id_transaccion` sea único y que los valores de `id_cliente` esten asociados a un cliente registrado.

### 👥 Diagnóstico de los datos de Clientes

In [3]:
print(SEPARATOR)
print("📊 REPORTE DE DIAGNÓSTICO: FRECUENCIAS Y VARIACIONES")
print(SEPARATOR)

# 1. Estado de Actividad
print(f"\n🔹 1. VARIACIONES HETEROGÉNEAS EN COLUMNA 'ACTIVO'\n{SUB_LINE}")
df_clientes_raw.groupBy("activo").count().sort(F.desc("count")).show(truncate=False)

# 2. Ubicación Geográfica
print(f"\n🔹 2. FRECUENCIA DE VALORES EN 'CIUDAD'\n{SUB_LINE}")
df_clientes_raw.groupBy("ciudad").count().sort(F.desc("count")).show(truncate=False)

# 3. Portafolio de Productos (Manejo de Arrays con Explode)
print(f"\n🔹 3. VALORES ÚNICOS DETECTADOS EN ARRAY 'PRODUCTOS' Y SU FRECUENCIA\n{SUB_LINE}")
df_productos_unificados = df_clientes_raw.select(F.explode_outer("productos").alias("producto_individual"))
df_productos_unificados.groupBy("producto_individual").count().sort(F.desc("count")).show(truncate=False)

# 4. Segmentación Comercial
print(f"\n🔹 4. FRECUENCIA DE VALORES EN 'SEGMENTO'\n{SUB_LINE}")
df_clientes_raw.groupBy("segmento").count().sort(F.desc("count")).show(truncate=False)

# 5. Identificación y Documentos
print(f"\n🔹 5. POSIBLES VALORES EN 'TIPO_DOCUMENTO'\n{SUB_LINE}")
df_clientes_raw.groupBy("tipo_documento").count().sort(F.desc("count")).show(truncate=False)

print(SEPARATOR)

📊 REPORTE DE DIAGNÓSTICO: FRECUENCIAS Y VARIACIONES

🔹 1. VARIACIONES HETEROGÉNEAS EN COLUMNA 'ACTIVO'
------------------------------------------------------------------------------------------
+------+-----+
|activo|count|
+------+-----+
|true  |11   |
|false |9    |
|SI    |8    |
|Si    |7    |
|NO    |6    |
|No    |5    |
|0     |4    |
|1     |2    |
+------+-----+


🔹 2. FRECUENCIA DE VALORES EN 'CIUDAD'
------------------------------------------------------------------------------------------
+------------+-----+
|ciudad      |count|
+------------+-----+
|cali        |8    |
|  Bogotá    |6    |
|medellín    |6    |
|b/quilla    |5    |
|Bogota      |5    |
|Pereira     |4    |
|Cali        |4    |
| Cúcuta     |4    |
|BOGOTA      |3    |
|Bucaramanga |2    |
|Barranquilla|2    |
|Manizales   |1    |
|Medellin    |1    |
|Cartagena   |1    |
+------------+-----+


🔹 3. VALORES ÚNICOS DETECTADOS EN ARRAY 'PRODUCTOS' Y SU FRECUENCIA
----------------------------------------------

In [4]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE INTEGRIDAD: IDENTIFICADORES Y DUPLICADOS")
print(SEPARATOR)

# ==========================================================================
# PASO 1: Validación de Calidad en id_cliente
# ==========================================================================
print(f"\n🔹 1. VALIDACIÓN DE 'id_cliente' y 'numero_documento'\n{SUB_LINE}")

df_clientes_raw.select(
    # Auditoría de IDs
    F.count(F.when(F.col("id_cliente").isNull() | (F.col("id_cliente") == ""), 1)).alias("IDs_Nulos_o_Vacios"),
    F.count(F.when(~F.col("id_cliente").rlike(r"^\d+$"), 1)).alias("IDs_No_Numericos"),
    
    # Auditoria de numeros de documento
    F.count(F.when(F.col("numero_documento").isNull() | (F.col("numero_documento") == ""), 1)).alias("Documentos_Nulos_o_Vacios")
).show()

# ==========================================================================
# PASO 2: Detección de Duplicados
# ==========================================================================
print(f"\n🔹 2. ANÁLISIS DE DUPLICADOS POR 'id_cliente'\n{SUB_LINE}")
# Agrupamos por ID y filtramos los que aparezcan más de una vez
df_duplicados_id = df_clientes_raw.groupBy("id_cliente").count().filter(F.col("count") > 1)

if df_duplicados_id.count() > 0:
    print(f"Cantidad de id_cliente repetidos: {df_duplicados_id.count()}")
    df_duplicados_id.show(truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron IDs duplicados en el archivo de clientes.")

print(f"\n🔹 3. ANÁLISIS DE DUPLICADOS POR 'numero_documento'\n{SUB_LINE}")
# Agrupamos por documento, ignorando temporalmente los nulos para no sesgar el conteo de repetidos
df_duplicados_doc = df_clientes_raw.filter(F.col("numero_documento").isNotNull() & (F.col("numero_documento") != ""))\
    .groupBy("numero_documento").count().filter(F.col("count") > 1)

if df_duplicados_doc.count() > 0:
    print(f"Cantidad de numero_documento repetidos: {df_duplicados_doc.count()}")
    df_duplicados_doc.show(truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron números de documento duplicados en el archivo de clientes.")


print(SEPARATOR)

🔍 AUDITORÍA DE INTEGRIDAD: IDENTIFICADORES Y DUPLICADOS

🔹 1. VALIDACIÓN DE 'id_cliente' y 'numero_documento'
------------------------------------------------------------------------------------------
+------------------+----------------+-------------------------+
|IDs_Nulos_o_Vacios|IDs_No_Numericos|Documentos_Nulos_o_Vacios|
+------------------+----------------+-------------------------+
|                 0|               0|                        5|
+------------------+----------------+-------------------------+


🔹 2. ANÁLISIS DE DUPLICADOS POR 'id_cliente'
------------------------------------------------------------------------------------------
Cantidad de id_cliente repetidos: 2
+----------+-----+
|id_cliente|count|
+----------+-----+
|1004      |2    |
|1011      |2    |
+----------+-----+


🔹 3. ANÁLISIS DE DUPLICADOS POR 'numero_documento'
------------------------------------------------------------------------------------------
Cantidad de numero_documento repetidos: 2
+----

### 📝 Hallazgos (Diagnóstico de los datos de Clientes)
*   En las columnas `activo`, `ciudad`, `segmento` y `tipo_documento`, las cuales funcionan como variables categóricas, es necesario estandarizar los valores.
*   Se identificaron filas con `id_cliente` y `numero_documento` repetidos, por lo que se debe explorar en detalle el registro completo para concluir si están duplicados.
*   Hay valores nulos en `numero_documento`, por lo que es necesario analizar en profundidad estos registros, para descartar que no estén duplicados, aunque tengan distinto `id_cliente`.
*   En los valores de `contacto.telefono`, `fecha_alta`, `nombre` y `numero_documento`, se encuentran disntintos formatos de escritura, por lo que se deben estandarizar.


### 💸 Diagnóstico de los datos de Transacciones

In [5]:
print(SEPARATOR)
print("📊 REPORTE DE DIAGNÓSTICO TRANSACCIONES: FRECUENCIAS Y VARIACIONES")
print(SEPARATOR)

# 1. Tipo de Producto
print(f"\n🔹 1. VARIACIONES EN COLUMNA 'tipo_producto'\n{SUB_LINE}")
df_transacciones_raw.groupBy("tipo_producto").count().sort(F.desc("count")).show(truncate=False)

# 2. Moneda de la Operación
print(f"\n🔹 2. FRECUENCIA DE VALORES EN 'moneda'\n{SUB_LINE}")
df_transacciones_raw.groupBy("moneda").count().sort(F.desc("count")).show(truncate=False)

# 3. Canal de la Transacción
print(f"\n🔹 3. FRECUENCIA DE VALORES EN 'canal'\n{SUB_LINE}")
df_transacciones_raw.groupBy("canal").count().sort(F.desc("count")).show(truncate=False)

# 4. Estado de la Transacción
print(f"\n🔹 4. FRECUENCIA DE VALORES EN 'estado'\n{SUB_LINE}")
df_transacciones_raw.groupBy("estado").count().sort(F.desc("count")).show(truncate=False)

# 5. Sucursal Origen
print(f"\n🔹 5. VARIACIONES EN COLUMNA 'sucursal'\n{SUB_LINE}")
df_transacciones_raw.groupBy("sucursal").count().sort(F.desc("count")).show(truncate=False)

print(SEPARATOR)

📊 REPORTE DE DIAGNÓSTICO TRANSACCIONES: FRECUENCIAS Y VARIACIONES

🔹 1. VARIACIONES EN COLUMNA 'tipo_producto'
------------------------------------------------------------------------------------------
+-----------------------+-----+
|tipo_producto          |count|
+-----------------------+-----+
|Cdt                    |40   |
|Cuenta de ahorros      |35   |
|CTA_AHORROS            |33   |
|Certificado de Deposito|33   |
|crédito                |31   |
|ahorros                |30   |
|CDT                    |29   |
|Cuenta Ahorros         |26   |
|CERTIFICADO DE DEPOSITO|23   |
|AHORROS                |23   |
|CORRIENTE              |22   |
|cdt                    |21   |
|cta corriente          |18   |
|CREDITO                |17   |
|Crédito                |15   |
|Credito                |13   |
|Cuenta corriente       |11   |
|Cuenta Corriente       |9    |
+-----------------------+-----+


🔹 2. FRECUENCIA DE VALORES EN 'moneda'
-----------------------------------------------------

In [6]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE INTEGRIDAD EN TRANSACCIONES: id_transaccion")
print(SEPARATOR)

# ==========================================================================
# PASO 1: Nulos, Vacíos y Formato Estructural
# ==========================================================================
print(f"\n🔹 1. VALIDACIÓN DE CALIDAD EN 'id_transaccion'\n{SUB_LINE}")

df_transacciones_raw.select(
    F.count(F.when(F.col("id_transaccion").isNull() | (F.col("id_transaccion") == ""), 1)).alias("TX_Nulas_o_Vacias"),
    F.count(F.when(~F.col("id_transaccion").rlike(r"^\d+$"), 1)).alias("TX_No_Numericas")
).show()

# ==========================================================================
# PASO 2: Detección de Duplicados Críticos
# ==========================================================================
print(f"\n🔹 2. ANÁLISIS DE REPETIDOS POR 'id_transaccion'\n{SUB_LINE}")

# Agrupamos por identificador de transacción y filtramos los que aparezcan más de una vez
df_duplicados_tx = df_transacciones_raw.groupBy("id_transaccion").count().filter(F.col("count") > 1)

print(f"Cantidad de id_transaccion duplicados: {df_duplicados_tx.count()}")

if df_duplicados_tx.count() > 0:
    print("\nMuestra de transacciones repetidas detectadas:")
    df_duplicados_tx.sort(F.desc("count")).show(10, truncate=False)
else:
    print("✨ ¡Perfecto! No se encontraron llaves duplicadas en el archivo de transacciones.")

print(SEPARATOR)

🔍 AUDITORÍA DE INTEGRIDAD EN TRANSACCIONES: id_transaccion

🔹 1. VALIDACIÓN DE CALIDAD EN 'id_transaccion'
------------------------------------------------------------------------------------------
+-----------------+---------------+
|TX_Nulas_o_Vacias|TX_No_Numericas|
+-----------------+---------------+
|                0|              0|
+-----------------+---------------+


🔹 2. ANÁLISIS DE REPETIDOS POR 'id_transaccion'
------------------------------------------------------------------------------------------
Cantidad de id_transaccion duplicados: 6

Muestra de transacciones repetidas detectadas:
+--------------+-----+
|id_transaccion|count|
+--------------+-----+
|500148        |2    |
|500363        |2    |
|500153        |2    |
|500060        |2    |
|500123        |2    |
|500260        |2    |
+--------------+-----+



In [7]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE CALIDAD: 'id_cliente' EN EL ARCHIVO DE TRANSACCIONES")
print(SEPARATOR)

# Evaluar la estructura del id_cliente en transacciones antes de limpiar
df_transacciones_raw.select(
    F.count(F.when(F.col("id_cliente").isNull() | (F.col("id_cliente") == ""), 1)).alias("IDs_Nulos_o_Vacios"),
    F.count(F.when(~F.col("id_cliente").rlike(r"^\d+$"), 1)).alias("IDs_No_Numericos")
).show()


# Muestra física de los IDs que están sucios para guardarla en el reporte
print("Muestra de registros con id_cliente mal formateados en Transacciones:")
df_transacciones_raw.filter(
    F.col("id_cliente").isNotNull() & 
    (F.col("id_cliente") != "") & 
    (~F.col("id_cliente").rlike(r"^\d+$"))
).select("id_transaccion", "id_cliente").show(5, truncate=False)

print(SEPARATOR)

🔍 AUDITORÍA DE CALIDAD: 'id_cliente' EN EL ARCHIVO DE TRANSACCIONES
+------------------+----------------+
|IDs_Nulos_o_Vacios|IDs_No_Numericos|
+------------------+----------------+
|                 0|              63|
+------------------+----------------+

Muestra de registros con id_cliente mal formateados en Transacciones:
+--------------+----------+
|id_transaccion|id_cliente|
+--------------+----------+
|500379        | 1033     |
|500347        | 1033     |
|500324        | 1025     |
|500340        | 1017     |
|500023        | 1013     |
+--------------+----------+
only showing top 5 rows


In [8]:
print(SEPARATOR)
print("🔗 EVALUACIÓN DE INTEGRIDAD REFERENCIAL (APLICANDO TRIM)")
print(SEPARATOR)

# Hacemos el left_anti join aplicando F.trim() en AMBAS tablas para asegurar un cruce limpio
df_huerfanas_reales = df_transacciones_raw.join(
    df_clientes_raw,
    F.trim(df_transacciones_raw["id_cliente"]) == F.trim(df_clientes_raw["id_cliente"]),
    how="left_anti"
)

total_huerfanas_reales = df_huerfanas_reales.count()
print(f"\n❌ Cantidad REAL de transacciones huérfanas (tras remover espacios): {total_huerfanas_reales}")

if total_huerfanas_reales > 0:
    # Agrupar las transacciones huérfanas por el ID de cliente (limpio) y contar cuántas operaciones tiene cada uno
    df_recuento_fantasmas = df_huerfanas_reales.groupBy(
        F.trim(F.col("id_cliente")).alias("id_cliente_fantasma")
    ).count().sort(F.desc("count"))
    
    # Mostrar cuántos códigos de cliente distintos no existen en el maestro
    print(f"❌ Número de IDs de clientes únicos que NO existen en el maestro: {df_recuento_fantasmas.count()}")
    print("\nDetalle de IDs fantasmas y el volumen de transacciones que afectaría cada uno:")
    df_recuento_fantasmas.show(truncate=False)


else:
    print("\n✨ ¡Excelente! Al limpiar los espacios con trim(), el 100% de las transacciones cruzaron con éxito.")

🔗 EVALUACIÓN DE INTEGRIDAD REFERENCIAL (APLICANDO TRIM)

❌ Cantidad REAL de transacciones huérfanas (tras remover espacios): 20
❌ Número de IDs de clientes únicos que NO existen en el maestro: 3

Detalle de IDs fantasmas y el volumen de transacciones que afectaría cada uno:
+-------------------+-----+
|id_cliente_fantasma|count|
+-------------------+-----+
|7777               |13   |
|8888               |4    |
|9999               |3    |
+-------------------+-----+



### 📝 Hallazgos (Diagnóstico de los datos de Transacciones)
*   En las columnas `tipo_producto`, `moneda`, `canal`, `estado` y `sucursal`, las cuales funcionan como variables categóricas, es necesario estandarizar los valores.
*   Se identificaron filas con `id_transaccion` repetidos, por lo que se debe explorar en detalle el registro completo para concluir si están duplicados.
* Se encontraron registros con `id_cliente` fantasma, es decir, que no están registrados en los datos de clientes.
*   En los valores de `fecha`, `monto`, `tasa_interes` y `plazo_dias`, se encuentran disntintos formatos de escritura, por lo que se deben estandarizar.

---

## 🧹 Paso 2: Limpieza y Estandarización

Teniendo en cuenta los hallazgos del paso anterior de Exploración, se procede a dejar cada columna consistente y usable

### 👥 Limpieza de la Tabla Clientes

#### Estandarización de `activo`, `ciudad`, `segmento`, `tipo_documento`

In [9]:
from pyspark.sql.types import BooleanType

print(f"🔹 1.Estandarizando las variables categóricas...\n{SUB_LINE}")

##############################################################################
# 1. Transformar activo
# Usamos F.when para mapear todas las variaciones conocidas a True, y por defecto asignar False.
# Además, forzamos el casteo a BooleanType para estandarizar el tipo de dato.
df_clientes_clean = df_clientes_raw.withColumn(
    "activo_clean",
    F.when(
        # Convertimos a minúsculas y quitamos espacios
        F.trim(F.lower(F.col("activo"))).isin("si", "1", "true"), 
        True
    )
    .otherwise(False) # Maneja "0", "no", "false"
    .cast(BooleanType())
)
# Estado limpio de activo
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'activo_clean' (Tipo: Boolean):\n{SUB_LINE}")
df_clientes_clean.groupBy("activo_clean").count().show(truncate=False)

##############################################################################
# 2. Transformar ciudad
# Normalizamos tíldes y espacios
ACCENTED_CHARS = "áéíóúñ"
CLEAN_CHARS    = "aeioun"

df_clientes_clean = df_clientes_clean.withColumn(
    "ciudad_clean",
    # Traducimos las tildes a sus vocales limpias
    F.translate(
        # Convertimos a mayúscula solo la inicial
        F.initcap(
            # Removemos espacios en blanco iniciales y finales
            F.trim(F.col("ciudad"))
        ),
        ACCENTED_CHARS,
        CLEAN_CHARS
    )
)

# Corregimos abreviaciones
df_clientes_clean = df_clientes_clean.withColumn(
    "ciudad_clean",
    F.when(F.col("ciudad_clean") == "B/quilla", "Barranquilla")
     .otherwise(F.col("ciudad_clean")) # Si no es ninguna de las anteriores, conserva el valor limpio
)

# Estado limpio de ciudad
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'ciudad_clean':\n{SUB_LINE}")
df_clientes_clean.groupBy("ciudad_clean").count().sort(F.desc("count")).show(truncate=False)

##############################################################################
# 3. Transformar segmento
# Mayúscula solo inicial y eliminamos espacios sobrantes
df_clientes_clean = df_clientes_clean.withColumn(
    "segmento_clean",
    F.initcap(
        F.trim(F.col("segmento"))
    )
)

# Aplicamos lógica condicional pde mayúsculas solo en la sigla PYME
df_clientes_clean = df_clientes_clean.withColumn(
    "segmento_clean",
    F.when(
        F.col("segmento_clean") == "Pyme", 
        "PYME"
    )
    # Para todo lo demás, aplicamos Title Case (Primera letra Mayúscula)
    .otherwise(F.col("segmento_clean"))
)

# Estado limpio de segmento
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'segmento_clean':\n{SUB_LINE}")
df_clientes_clean.groupBy("segmento_clean").count().sort(F.desc("count")).show(truncate=False)

##############################################################################
# 4. Transformar tipo_documento
df_clientes_clean = df_clientes_clean.withColumn(
    "tipo_documento_clean",
    # Reemplazamos todos los puntos "." por un string vacío ""
    # Nota: El punto en Regex representa "cualquier carácter", así que lo escapamos con "\"
    F.regexp_replace(
        # Convertimos a mayúsculas
        F.upper(
            # Removemos espacios exteriores
            F.trim(F.col("tipo_documento"))
        ),
        r"\.", 
        ""
    )
)

# Estado limpio de tipo_documento
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'tipo_documento_clean':\n{SUB_LINE}")
df_clientes_clean.groupBy("tipo_documento_clean").count().sort(F.desc("count")).show(truncate=False)

🔹 1.Estandarizando las variables categóricas...
------------------------------------------------------------------------------------------

🔹 Frecuencias ESTANDARIZADAS en 'activo_clean' (Tipo: Boolean):
------------------------------------------------------------------------------------------
+------------+-----+
|activo_clean|count|
+------------+-----+
|true        |28   |
|false       |24   |
+------------+-----+


🔹 Frecuencias ESTANDARIZADAS en 'ciudad_clean':
------------------------------------------------------------------------------------------
+------------+-----+
|ciudad_clean|count|
+------------+-----+
|Bogota      |14   |
|Cali        |12   |
|Medellin    |7    |
|Barranquilla|7    |
|Pereira     |4    |
|Cucuta      |4    |
|Bucaramanga |2    |
|Manizales   |1    |
|Cartagena   |1    |
+------------+-----+


🔹 Frecuencias ESTANDARIZADAS en 'segmento_clean':
------------------------------------------------------------------------------------------
+--------------+-----+

#### Limpieza de `nombre`

In [10]:
print(SEPARATOR)
print("🔍 AUDITORÍA DE CALIDAD: DETECCIÓN DE CARACTERES RAROS EN NOMBRES")
print(SEPARATOR)

# Definimos el patrón de caracteres PERMITIDOS en un nombre en español.
patron_caracteres_raros = r"[^a-zA-ZáéíóúüñÁÉÍÓÚÜÑ\s\.\,\-]"

# Filtramos el DataFrame para encontrar filas que contengan cualquier carácter NO permitido
df_sospechosos = df_clientes_clean.filter(
    F.col("nombre").rlike(patron_caracteres_raros)
)

total_sospechosos = df_sospechosos.count()
print(f"⚠️ Se encontraron {total_sospechosos} registros con posibles caracteres corruptos.\n")

if total_sospechosos > 0:
    print("Muestra de casos detectados (Revisa si hay tildes rotas o codificaciones dañadas):")
    df_sospechosos.select("id_cliente", "nombre").show(15, truncate=False)
else:
    print("✨ ¡Excelente! No se detectaron caracteres raros ni tildes rotas en la columna 'nombre'.")
print(SEPARATOR)

🔍 AUDITORÍA DE CALIDAD: DETECCIÓN DE CARACTERES RAROS EN NOMBRES
⚠️ Se encontraron 0 registros con posibles caracteres corruptos.

✨ ¡Excelente! No se detectaron caracteres raros ni tildes rotas en la columna 'nombre'.


In [11]:
print(f"🔹 2.Arreglando heterogeneidad en la escritura...\n{SUB_LINE}")

##############################################################################
# 1. Limpiar columna nombre
print(f"\n🔹 Limpiando la columna nombre...")
# Aplicar la limpieza respetando acentos, tildes y puntos.
df_clientes_clean = df_clientes_clean.withColumn(
    "nombre_clean",
    # Reemplazar espacios múltiples intermedios por un solo espacio
    F.regexp_replace(
        # Eliminar espacios al inicio y al final
        F.trim(F.col("nombre")),
        r"\s+", 
        " "
    )
)

# Convertir a solo mayúscula inicial
df_clientes_clean = df_clientes_clean.withColumn("nombre_clean", F.initcap(F.col("nombre_clean")))


# Mostrar resultado a modo de ejemplo
print(f"🔹 Muestra de nombres limpios:")
df_clientes_clean.select("id_cliente", "nombre", "nombre_clean").show(10, truncate=False)



🔹 2.Arreglando heterogeneidad en la escritura...
------------------------------------------------------------------------------------------

🔹 Limpiando la columna nombre...
🔹 Muestra de nombres limpios:
+----------+------------------+------------------+
|id_cliente|nombre            |nombre_clean      |
+----------+------------------+------------------+
|1001      |María José Álvarez|María José Álvarez|
|1002      |Daniela Suarez    |Daniela Suarez    |
|1003      |Ana Maria Rojas   |Ana Maria Rojas   |
|1004      |Isabella Diaz     |Isabella Diaz     |
|1005      |Jorge  Mendoza    |Jorge Mendoza     |
|1006      |SANTIAGO VARGAS   |Santiago Vargas   |
|1007      |SANTIAGO VARGAS   |Santiago Vargas   |
|1008      |LUCIA FERNANDEZ   |Lucia Fernandez   |
|1009      |Gabriela Nino     |Gabriela Nino     |
|1010      |Gabriela Nino     |Gabriela Nino     |
+----------+------------------+------------------+
only showing top 10 rows


#### Limpieza de `contacto`

In [12]:
# 2. Limpieza inicial de la columna contacto (telefono y email)
print(f"\n🔹 Limpiando la columna contacto (email y teléfono)...")

# Aplicamos trasformacion al campo interno email
# Limpiamos los espacios al inicio y al final
df_clientes_clean = df_clientes_clean.withColumn(
    "contacto_clean",
    F.col("contacto")
    .withField("email", F.trim(F.col("contacto.email")))
    .withField("telefono", F.trim(F.col("contacto.telefono")))
)

##############################################################################
print(SEPARATOR)
print("🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'contacto.telefono'")
print(SEPARATOR)

# Creamos la columna de residuo aplicando TRIM primero (por eficiencia) y luego removiendo números y espacios
df_diagnostico = df_clientes_clean.withColumn(
    "solo_ruido",
    F.regexp_replace(F.trim(F.col("contacto_clean.telefono")), r"[0-9\s]", "")
)

# Le damos una etiqueta visible a los vacíos para que no se confundan en el .show()
df_diagnostico = df_diagnostico.withColumn(
    "solo_ruido_visible",
    F.when(F.col("solo_ruido") == "", "[STRING VACÍO]")
     .otherwise(F.col("solo_ruido"))
)

# Agrupamos por los residuos encontrados
print(f"🔹 Residuos no numéricos consolidados:")
df_diagnostico.groupBy("solo_ruido_visible") \
    .count() \
    .sort(F.desc("count")) \
    .show(20, truncate=False)

##############################################################################
print(SEPARATOR)
print("🔬 AUDITORÍA DE INDICATIVOS INTERNACIONALES (¿Todo es +57?)")
print(SEPARATOR)

# Filtramos los teléfonos que inician con el símbolo "+"
# Usamos F.trim para asegurar que evaluamos el inicio real del string
df_con_mas = df_clientes_clean.filter(F.trim(F.col("contacto_clean.telefono")).startswith("+"))

# Extraemos los primeros 3 caracteres (ej: '+57') para agruparlos
df_indicativos = df_con_mas.withColumn(
    "prefijo_pais", 
    F.substring(F.trim(F.col("contacto_clean.telefono")), 1, 3)
)

print(f"🔹 Conteo y frecuencia de prefijos encontrados:")
df_indicativos.groupBy("prefijo_pais") \
    .count() \
    .sort(F.desc("count")) \
    .show(truncate=False)

##############################################################################
# Aplicamos la transformación definitiva al campo interno telefono
df_clientes_clean = df_clientes_clean.withColumn(
    "contacto_clean",
    F.col("contacto_clean").withField(
        "telefono",
        # El patrón busca remover el texto '+57' o cualquiera de los caracteres: (, ), espacios o guiones
        F.regexp_replace(F.col("contacto_clean.telefono"), r"(\+57|[\(\)\s\-])", "")
    )
)

print(SEPARATOR)
print("🔬 AUDITORÍA FINAL DE LONGITUD DE TELÉFONOS (META: 10 DÍGITOS)")
print(SEPARATOR)

# Crear columna temporal con la longitud y la clasificación del estado
df_auditoria = df_clientes_clean.withColumn(
    "longitud", 
    F.length(F.col("contacto_clean.telefono"))
).withColumn(
    "estado_telefono",
    F.when(
        F.col("contacto_clean.telefono").isNull() | (F.col("contacto_clean.telefono") == ""),
        "NULL")
     .when(F.col("longitud") == 10, "OK (10 Dígitos)")
     .otherwise(F.concat(F.lit("ERROR ("), F.col("longitud"), F.lit(" dígitos)")))
)

# Mostrar el consolidado general para ver cuántos registros válidos y erróneos hay
print(f"🔹 Resumen final de estados de longitud:")
df_auditoria.groupBy("estado_telefono") \
    .count() \
    .sort(F.desc("count")) \
    .show(truncate=False)


🔹 Limpiando la columna contacto (email y teléfono)...
🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'contacto.telefono'
🔹 Residuos no numéricos consolidados:
+------------------+-----+
|solo_ruido_visible|count|
+------------------+-----+
|[STRING VACÍO]    |21   |
|+                 |14   |
|NULL              |9    |
|()                |8    |
+------------------+-----+

🔬 AUDITORÍA DE INDICATIVOS INTERNACIONALES (¿Todo es +57?)
🔹 Conteo y frecuencia de prefijos encontrados:
+------------+-----+
|prefijo_pais|count|
+------------+-----+
|+57         |14   |
+------------+-----+

🔬 AUDITORÍA FINAL DE LONGITUD DE TELÉFONOS (META: 10 DÍGITOS)
🔹 Resumen final de estados de longitud:
+---------------+-----+
|estado_telefono|count|
+---------------+-----+
|OK (10 Dígitos)|36   |
|NULL           |16   |
+---------------+-----+



#### Limpieza de `fecha_alta`

In [13]:
print(SEPARATOR)
print("🔬 DIAGNÓSTICO MAESTRO: ANÁLISIS DE FORMATOS Y DESEMPATE CRUZADO EN 'fecha_alta'")
print(SEPARATOR)

##############################################################################
# Análisis de Formatos
# Definición de Expresiones Regulares posibles
regex_iso = r"^\d{4}-\d{2}-\d{2}$"            # Ej: 2026-07-13 AAAA-##-##
regex_barra = r"^\d{1,2}/\d{1,2}/\d{4}$"    # Ej: 13/07/2026 ##/##/AAAA
regex_guion = r"^\d{1,2}-\d{1,2}-\d{4}$"    # Ej: 13-07-2026 ##-##-AAAA
regex_texto_mmm = r"^\d{1,2}-[a-zA-Z]{3}-\d{2,4}$"  # Ej: 13-Jul-2026 DD-mmm-AAA

# Clasificación en el DataFrame
df_formatos = df_clientes_clean.withColumn(
    "formato_detectado",
    F.when(F.col("fecha_alta").isNull() | (F.col("fecha_alta") == ""), "NULL")
     .when(F.col("fecha_alta").rlike(regex_iso), "YYYY-MM-DD (ISO)")
     .when(F.col("fecha_alta").rlike(regex_barra), "DD/MM/YYYY o MM/DD/YYYY (Barras)")
     .when(F.col("fecha_alta").rlike(regex_guion), "DD-MM-YYYY o MM-DD-YYYY (Guiones)")
     .when(F.col("fecha_alta").rlike(regex_texto_mmm), "DD-mmm-AAAA (Mes abreviado)")
     .otherwise(F.concat(F.lit("OTRO / DESCONOCIDO: '"), F.col("fecha_alta"), F.lit("'")))
)

# Mostrar el consolidado general de frecuencias
print(f"🔹 Frecuencia de formatos detectados en la columna:")
df_formatos.groupBy("formato_detectado") \
    .count() \
    .sort(F.desc("count")) \
    .show(truncate=False)

##############################################################################
# AUDITORÍA CRUZADA
print(f"💡 MATRIZ DE DESEMPATE ANALÍTICO (Posición 1 vs Posición 2):")

# Filtramos los formatos numéricos ambiguos
df_ambiguos = df_formatos.filter(
    F.col("formato_detectado").isin(
        "DD/MM/YYYY o MM/DD/YYYY (Barras)", 
        "DD-MM-YYYY o MM-DD-YYYY (Guiones)"
    )
)

# Separamos los componentes por guion o barra y los convertimos a enteros
df_posiciones = df_ambiguos.withColumn(
    "partes", F.split(F.col("fecha_alta"), "[-/]")
).withColumn(
    "pos_1", F.col("partes").getItem(0).cast("int")
).withColumn(
    "pos_2", F.col("partes").getItem(1).cast("int")
)

# Agrupamos y calculamos los máximos y banderas de control para ambas posiciones
df_posiciones.groupBy("formato_detectado") \
 .agg(
     F.max("pos_1").alias("Max_Posicion_1"),
     F.count(F.when(F.col("pos_1") > 12, 1)).alias("Casos_Pos1_>_12"),
     F.max("pos_2").alias("Max_Posicion_2"),
     F.count(F.when(F.col("pos_2") > 12, 1)).alias("Casos_Pos2_>_12")
 ).show(truncate=False)

##############################################################################
print(f"💡 MATRIZ DE DESEMPATE PARA REGISTROS ISO (¿AAAA-MM-DD o AAAA-DD-MM?):")

# Filtramos únicamente los registros que inicialmente clasificaron como ISO
df_iso = df_formatos.filter(F.col("formato_detectado") == "YYYY-MM-DD (ISO)")

# Separamos por guion. Index 0 = Año, Index 1 = Posición Media, Index 2 = Posición Final
df_iso_posiciones = df_iso.withColumn(
    "partes_iso", F.split(F.col("fecha_alta"), "-")
).withColumn(
    "pos_medio", F.col("partes_iso").getItem(1).cast("int")  # Debería ser el Mes (1-12)
).withColumn(
    "pos_final", F.col("partes_iso").getItem(2).cast("int")  # Debería ser el Día (1-31)
)

# Agrupamos para calcular los máximos y detectar si el medio supera el mes 12
df_iso_posiciones.agg(
    F.max("pos_medio").alias("Max_Posicion_Medio (Mes?)"),
    F.count(F.when(F.col("pos_medio") > 12, 1)).alias("Casos_Medio_>_12 (Alerta AAAA-DD-MM)"),
    F.max("pos_final").alias("Max_Posicion_Final (Día?)"),
    F.count(F.when(F.col("pos_final") > 12, 1)).alias("Casos_Final_>_12 (Valida AAAA-MM-DD)")
).show(truncate=False)

🔬 DIAGNÓSTICO MAESTRO: ANÁLISIS DE FORMATOS Y DESEMPATE CRUZADO EN 'fecha_alta'
🔹 Frecuencia de formatos detectados en la columna:
+---------------------------------+-----+
|formato_detectado                |count|
+---------------------------------+-----+
|DD-MM-YYYY o MM-DD-YYYY (Guiones)|14   |
|DD/MM/YYYY o MM/DD/YYYY (Barras) |14   |
|YYYY-MM-DD (ISO)                 |12   |
|DD-mmm-AAAA (Mes abreviado)      |12   |
+---------------------------------+-----+

💡 MATRIZ DE DESEMPATE ANALÍTICO (Posición 1 vs Posición 2):
+---------------------------------+--------------+---------------+--------------+---------------+
|formato_detectado                |Max_Posicion_1|Casos_Pos1_>_12|Max_Posicion_2|Casos_Pos2_>_12|
+---------------------------------+--------------+---------------+--------------+---------------+
|DD-MM-YYYY o MM-DD-YYYY (Guiones)|28            |10             |12            |0              |
|DD/MM/YYYY o MM/DD/YYYY (Barras) |28            |7              |12            

In [14]:
# 3. Limpieza de la columna fecha_alta
print(f"\n🔹 Limpiando la columna fecha_alta...")

##############################################################################
# Creamos una columna temporal con el texto de la fecha estandarizado en Minúsculas
# para reemplazar los meses conflictivos de español a inglés.
df_fecha_traducida = df_clientes_clean.withColumn(
    "fecha_alta_temp",
    F.lower(F.col("fecha_alta"))
).withColumn(
    "fecha_alta_temp",
    F.regexp_replace(F.col("fecha_alta_temp"), r"-ene-", "-jan-")
).withColumn(
    "fecha_alta_temp",
    F.regexp_replace(F.col("fecha_alta_temp"), r"-abr-", "-apr-")
).withColumn(
    "fecha_alta_temp",
    F.regexp_replace(F.col("fecha_alta_temp"), r"-ago-", "-aug-")
).withColumn(
    "fecha_alta_temp",
    F.regexp_replace(F.col("fecha_alta_temp"), r"-dic-", "-dec-")
)

##############################################################################
#  APLICAR PIPELINE DE CONVERSIÓN CON COALESCE
# Se evalúan en cascada. El primero que logre parsear la fecha define el valor de la fila.
df_clientes_clean = df_fecha_traducida.withColumn(
    "fecha_alta_clean",
    F.coalesce(
        F.try_to_date(F.col("fecha_alta"), "yyyy-MM-dd"),       # 1. Estándar ISO
        F.try_to_date(F.col("fecha_alta"), "dd/MM/yyyy"),       # 2. Latam con barras
        F.try_to_date(F.col("fecha_alta"), "dd-MM-yyyy"),       # 3. Latam con guiones
        # Para el de texto, usamos la columna temporal
        F.try_to_date(F.col("fecha_alta_temp"), "dd-MMM-yyyy")  # 4. Mes literal (ej: 13-jul-2026)
    )
).drop("fecha_alta_temp")

##############################################################################
# MOSTRAR MUESTRA FINAL (Después - Todo unificado en yyyy-MM-dd)
print(f"\n🔹 Muestra de fechas NORMALIZADAS (Tipo DateType nativo de Spark):")
df_clientes_clean.select("id_cliente", "fecha_alta_clean").show(10, truncate=False)

# 4. AUDITORÍA DE CALIDAD: Asegurar que no generamos nulos nuevos por error de casteo
# Contamos cuántos nulos/vacíos había antes vs cuántos hay ahora
print(f"\n🔬 Control de calidad post-casteo:")
print(SUB_LINE)
total_nulos_final = df_clientes_clean.filter(F.col("fecha_alta_clean").isNull()).count()
print(f"✅ Conversión finalizada. Total de registros que quedaron como NULL: {total_nulos_final}")


🔹 Limpiando la columna fecha_alta...

🔹 Muestra de fechas NORMALIZADAS (Tipo DateType nativo de Spark):
+----------+----------------+
|id_cliente|fecha_alta_clean|
+----------+----------------+
|1001      |2023-10-22      |
|1002      |2021-02-02      |
|1003      |2020-07-01      |
|1004      |2022-01-07      |
|1005      |2022-08-09      |
|1006      |2022-07-10      |
|1007      |2021-07-11      |
|1008      |2022-08-07      |
|1009      |2021-08-18      |
|1010      |2023-12-02      |
+----------+----------------+
only showing top 10 rows

🔬 Control de calidad post-casteo:
------------------------------------------------------------------------------------------
✅ Conversión finalizada. Total de registros que quedaron como NULL: 0


#### Limpieza de `numero_documento`

In [15]:
# 4. Limpieza de la columna numero_documento
print(f"\n🔹 Limpiando la columna numero_documento...")

# Limpiamos espacios al inicio y al final
df_clientes_clean = df_clientes_clean.withColumn(
    "numero_documento_clean",
    F.trim(F.col("numero_documento"))
)

##############################################################################
print(SEPARATOR)
print("🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'numero_documento_clean'")
print(SEPARATOR)

# Creamos la columna de residuo aplicando TRIM primero (por eficiencia) y luego removiendo números y espacios
df_diagnostico = df_clientes_clean.withColumn(
    "solo_ruido",
    # F.regexp_replace(F.trim(F.col("numero_documento")), r"[0-9\s]", "")
    F.regexp_replace(F.col("numero_documento_clean"), r"[0-9]", "")
)

# Le damos una etiqueta visible a los vacíos para que no se confundan en el .show()
df_diagnostico = df_diagnostico.withColumn(
    "solo_ruido_visible",
    F.when(F.col("solo_ruido") == "", "[STRING VACÍO]")
     .otherwise(F.col("solo_ruido"))
)

# Agrupamos por los residuos encontrados
print(f"🔹 Residuos no numéricos consolidados:")
df_diagnostico.groupBy("solo_ruido_visible") \
    .count() \
    .sort(F.desc("count")) \
    .show(20, truncate=False)

##############################################################################
print(SEPARATOR)
print("🔬 COMPROBACIÓN: ¿EL GUION ES EXCLUSIVO DE LOS REGISTROS CON 'NIT'?")
print(SEPARATOR)

# 1. Creamos una columna temporal booleana para saber si tiene guion
df_validacion_docs = df_clientes_clean.withColumn(
    "tiene_guion", 
    F.col("numero_documento_clean").contains("-")
)

# 2. Agrupamos por tipo_documento y tiene_guion para ver la distribución total
print(f"🔹 Distribución de guiones por Tipo de Documento:")
df_validacion_docs.groupBy("tipo_documento_clean", "tiene_guion") \
    .count() \
    .sort("tipo_documento_clean", F.desc("tiene_guion")) \
    .show(truncate=False)


# Se eliminan solo los puntos
df_clientes_clean = df_clientes_clean.withColumn(
    "numero_documento_clean",
    F.replace(F.col("numero_documento"), F.lit("."), F.lit(""))
)


🔹 Limpiando la columna numero_documento...
🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'numero_documento_clean'
🔹 Residuos no numéricos consolidados:
+------------------+-----+
|solo_ruido_visible|count|
+------------------+-----+
|[STRING VACÍO]    |25   |
|..                |12   |
|..-               |10   |
|NULL              |5    |
+------------------+-----+

🔬 COMPROBACIÓN: ¿EL GUION ES EXCLUSIVO DE LOS REGISTROS CON 'NIT'?
🔹 Distribución de guiones por Tipo de Documento:
+--------------------+-----------+-----+
|tipo_documento_clean|tiene_guion|count|
+--------------------+-----------+-----+
|NULL                |NULL       |5    |
|CC                  |false      |21   |
|NIT                 |true       |10   |
|NIT                 |false      |16   |
+--------------------+-----------+-----+



#### Limpieza de `id_cliente`

In [16]:
# 5. Limpieza de la columna id_cliente
print(f"\n🔹 Limpiando y validando la columna id_cliente...")

# Aplicamos trim y validamos que sea un número entero puro directamente
df_clientes_clean = df_clientes_clean.withColumn(
    "id_cliente_clean",
    F.when(
        F.col("id_cliente").isNull() | (F.trim(F.col("id_cliente")) == ""), 
        F.lit(None)
    )
    .when(
        # Validamos que consista solo de dígitos
        F.trim(F.col("id_cliente")).rlike(r"^\d+$"),
        F.trim(F.col("id_cliente")).cast("long")
    )
    .otherwise(F.lit(None)) # Cualquier ID con letras o caracteres especiales pasa a NULL
)

# --- AUDITORÍA DE CALIDAD ---
print("📋 Muestra de IDs de cliente procesados:")
df_clientes_clean.select("id_cliente", "id_cliente_clean").show(10, truncate=False)

# Contamos cuántos registros tenían caracteres corruptos (letras/símbolos), excluyendo vacíos
clientes_corruptos = df_clientes_clean.filter(
    F.col("id_cliente_clean").isNull() & 
    F.col("id_cliente").isNotNull() & 
    (F.col("id_cliente") != "")
).count()

print(f"✅ Validación finalizada. Cuentas de clientes corruptas enviadas a NULL: {clientes_corruptos}")


🔹 Limpiando y validando la columna id_cliente...
📋 Muestra de IDs de cliente procesados:
+----------+----------------+
|id_cliente|id_cliente_clean|
+----------+----------------+
|1001      |1001            |
|1002      |1002            |
|1003      |1003            |
|1004      |1004            |
|1005      |1005            |
|1006      |1006            |
|1007      |1007            |
|1008      |1008            |
|1009      |1009            |
|1010      |1010            |
+----------+----------------+
only showing top 10 rows
✅ Validación finalizada. Cuentas de clientes corruptas enviadas a NULL: 0


#### Consolidación de tabla clientes

In [17]:
print(SEPARATOR)
print("✨ DEPURACIÓN FINAL: CONSTRUYENDO EL DATAFRAME CLIENTES LIMPIO")
print(SEPARATOR)

# Creamos el DataFrame definitivo seleccionando y renombrando en un solo viaje
df_clientes_final = df_clientes_clean.select(
    F.col("id_cliente_clean").alias("id_cliente"),
    F.col("nombre_clean").alias("nombre"),
    F.col("tipo_documento_clean").alias("tipo_documento"),
    F.col("numero_documento_clean").alias("numero_documento"),
    F.col("contacto_clean").alias("contacto"),  # Mantiene el struct limpio interno
    F.col("ciudad_clean").alias("ciudad"),
    F.col("segmento_clean").alias("segmento"),
    F.col("fecha_alta_clean").alias("fecha_alta"),
    F.col("activo_clean").alias("activo"),
    F.col("productos") # Esta no requería transformación
)

# Mostramos el nuevo esquema limpio
print("📋 Esquema final de la tabla 'clientes_final':")
df_clientes_final.printSchema()

print(f"🔹 Muestra de los datos listos:")
df_clientes_final.show(5, truncate=False)
print(SEPARATOR)

✨ DEPURACIÓN FINAL: CONSTRUYENDO EL DATAFRAME CLIENTES LIMPIO
📋 Esquema final de la tabla 'clientes_final':
root
 |-- id_cliente: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- contacto: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- telefono: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- activo: boolean (nullable = false)
 |-- productos: array (nullable = true)
 |    |-- element: string (containsNull = true)

🔹 Muestra de los datos listos:
+----------+------------------+--------------+----------------+--------------------------------------------+------------+--------+----------+------+----------------------------------+
|id_cliente|nombre            |tipo_documento|numero_documento|contacto                                    |ciudad      |segme

### 💸 Limpieza de la Tabla Transacciones

#### Estandarización de `tipo_producto`, `moneda`, `canal`, `estado` y `sucursal`

In [18]:
print(f"🔹 1.Estandarizando las variables categóricas...\n{SUB_LINE}")

##############################################################################
# 1. Transformar tipo_producto
# Lo dejamos con el mismo formato encontrado en clientes
df_transacciones_clean = df_transacciones_raw.withColumn(
    "tipo_producto_clean",
    F.when(
        F.lower(F.col("tipo_producto")).contains("ahorros"), 
        F.lit("Cuenta Ahorros")
    )
    .when(
        F.lower(F.col("tipo_producto")).contains("cred") | F.lower(F.col("tipo_producto")).contains("créd"), 
        F.lit("Credito")
    )
    .when(
        F.lower(F.col("tipo_producto")).contains("corriente"), 
        F.lit("Cuenta Corriente")
    )
    .when(
        F.lower(F.col("tipo_producto")).contains("cdt") | F.lower(F.col("tipo_producto")).contains("certificado de deposito"), 
        F.lit("CDT")
    )
    .otherwise(F.lit("Otro / No Identificado")) # Red de seguridad para datos corruptos
)

# Estado limpio de tipo_producto
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'tipo_producto_clean':\n{SUB_LINE}")
df_transacciones_clean.groupBy("tipo_producto_clean").count().show(truncate=False)

##############################################################################
# 2. Transformar moneda
df_transacciones_clean = df_transacciones_clean.withColumn(
    "moneda_clean",
    F.when(
        F.lower(F.col("moneda")).contains("usd") | F.lower(F.col("moneda")).contains("dolar") | F.lower(F.col("moneda")).contains("dólar"),
        F.lit("USD")
    )
    .when(
        F.lower(F.col("moneda")).contains("cop") | F.lower(F.col("moneda")).contains("peso") | F.lower(F.col("moneda")).contains("$"),
        F.lit("COP")
    )
    .otherwise(F.lit("NULL")) # Red de seguridad para valores vacíos o extraños
)

# Estado limpio de moneda
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'moneda_clean':\n{SUB_LINE}")
df_transacciones_clean.groupBy("moneda_clean").count().show(truncate=False)

##############################################################################
# 3. Transformar canal
# Estandarizamos las variaciones de canales a las 4 categorías oficiales
df_transacciones_clean = df_transacciones_clean.withColumn(
    "canal_clean",
    F.when(
        F.lower(F.col("canal")).contains("app"), 
        F.lit("App")
    )
    .when(
        F.lower(F.col("canal")).contains("web"), 
        F.lit("Web")
    )
    .when(
        F.lower(F.col("canal")).contains("oficina"), 
        F.lit("Oficina")
    )
    .when(
        F.lower(F.col("canal")).contains("cajero") | F.lower(F.col("canal")).contains("atm"), 
        F.lit("Cajero")
    )
    .otherwise(F.lit("NULL")) # Red de seguridad
)

# Estado limpio de canal
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'canal_clean':\n{SUB_LINE}")
df_transacciones_clean.groupBy("canal_clean").count().show(truncate=False)

##############################################################################
# 4. Transformar estado
# Estandarizamos variaciones de éxito, rechazo o estados intermedios
df_transacciones_clean = df_transacciones_clean.withColumn(
    "estado_clean",
    F.when(
        F.lower(F.col("estado")).contains("aprob"), 
        F.lit("Aprobada")
    )
    .when(
        F.lower(F.col("estado")).contains("rechaz"), 
        F.lit("Rechazada")
    )
    .when(
        F.lower(F.col("estado")).contains("pend") | F.lower(F.col("estado")).contains("proceso"), 
        F.lit("Pendiente")
    )
    .otherwise(F.lit("NULL")) # Red de seguridad
)

# Estado limpio de estado
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'estado_clean':\n{SUB_LINE}")
df_transacciones_clean.groupBy("estado_clean").count().show(truncate=False)

##############################################################################
# 5. Transformar sucursal
# Normalizamos tíldes y espacios
ACCENTED_CHARS = "áéíóúñ"
CLEAN_CHARS    = "aeioun"

df_transacciones_clean = df_transacciones_clean.withColumn(
    "sucursal_clean",
    # Traducimos las tildes a sus vocales limpias
    F.translate(
        # Convertimos a mayúscula solo la inicial
        F.initcap(
            # Removemos espacios en blanco iniciales y finales
            F.trim(F.col("sucursal"))
        ),
        ACCENTED_CHARS,
        CLEAN_CHARS
    )
)

# Corregimos abreviaciones
df_transacciones_clean = df_transacciones_clean.withColumn(
    "sucursal_clean",
    F.when(F.col("sucursal_clean") == "B/quilla", "Barranquilla")
     .otherwise(F.col("sucursal_clean")) # Si no es ninguna de las anteriores, conserva el valor limpio
)

# Estado limpio de ciudad
print(f"\n🔹 Frecuencias ESTANDARIZADAS en 'sucursal_clean':\n{SUB_LINE}")
df_transacciones_clean.groupBy("sucursal_clean").count().sort(F.desc("count")).show(truncate=False)

🔹 1.Estandarizando las variables categóricas...
------------------------------------------------------------------------------------------

🔹 Frecuencias ESTANDARIZADAS en 'tipo_producto_clean':
------------------------------------------------------------------------------------------
+-------------------+-----+
|tipo_producto_clean|count|
+-------------------+-----+
|CDT                |146  |
|Credito            |76   |
|Cuenta Ahorros     |147  |
|Cuenta Corriente   |60   |
+-------------------+-----+


🔹 Frecuencias ESTANDARIZADAS en 'moneda_clean':
------------------------------------------------------------------------------------------
+------------+-----+
|moneda_clean|count|
+------------+-----+
|COP         |332  |
|USD         |44   |
|NULL        |53   |
+------------+-----+


🔹 Frecuencias ESTANDARIZADAS en 'canal_clean':
------------------------------------------------------------------------------------------
+-----------+-----+
|canal_clean|count|
+-----------+-----+
|C

#### Limpieza de `fecha `

In [19]:
print(SEPARATOR)
print("🔬 DIAGNÓSTICO MAESTRO: ANÁLISIS DE FORMATOS Y DESEMPATE CRUZADO EN 'fecha'")
print(SEPARATOR)

##############################################################################
# Análisis de Formatos
# Definición de Expresiones Regulares posibles
regex_iso_guion = r"^\d{4}-\d{2}-\d{2}$"            # Ej: 2026-07-13 AAAA-MM-DD
regex_iso_barra = r"^\d{4}/\d{1,2}/\d{1,2}$"         # Ej: 2026/07/13 AAAA/MM/DD
regex_barra     = r"^\d{1,2}/\d{1,2}/\d{4}$"        # Ej: 13/07/2026 ##/##/AAAA
regex_guion     = r"^\d{1,2}-\d{1,2}-\d{4}$"        # Ej: 13-07-2026 ##-##-AAAA
regex_punto     = r"^\d{1,2}\.\d{1,2}\.\d{4}$"      # Ej: 13.07.2026 ##.##.AAAA
regex_texto_mmm = r"^\d{1,2}-[a-z]{3}-\d{2,4}$"     # Ej: 13-Jul-2026 DD-mmm-AAA

# Clasificación en el DataFrame
df_formatos = df_transacciones_clean.withColumn(
    "formato_detectado",
    F.when(F.col("fecha").isNull() | (F.col("fecha") == ""), "NULL")
     .when(F.col("fecha").rlike(regex_iso_guion) | F.col("fecha").rlike(regex_iso_barra), "YYYY-MM-DD (ISO / Estilo ISO)")
     .when(F.col("fecha").rlike(regex_barra), "DD/MM/YYYY o MM/DD/YYYY (Barras)")
     .when(F.col("fecha").rlike(regex_guion), "DD-MM-YYYY o MM-DD-YYYY (Guiones)")
     .when(F.col("fecha").rlike(regex_punto), "DD.MM.YYYY o MM.DD.YYYY (Puntos)")
     .when(F.col("fecha").rlike(regex_texto_mmm), "DD-mmm-AAAA (Mes abreviado)")
     .otherwise(F.concat(F.lit("OTRO / DESCONOCIDO: '"), F.col("fecha"), F.lit("'")))
)

# Mostrar el consolidado general de frecuencias
print(f"🔹 Frecuencia de formatos detectados en la columna:")
df_formatos.groupBy("formato_detectado") \
    .count() \
    .sort(F.desc("count")) \
    .show(truncate=False)

##############################################################################
# AUDITORÍA CRUZADA
print(f"💡 MATRIZ DE DESEMPATE ANALÍTICO (Posición 1 vs Posición 2):")

# Filtramos los formatos numéricos ambiguos (ahora incluye Puntos)
df_ambiguos = df_formatos.filter(
    F.col("formato_detectado").isin(
        "DD/MM/YYYY o MM/DD/YYYY (Barras)", 
        "DD-MM-YYYY o MM-DD-YYYY (Guiones)",
        "DD.MM.YYYY o MM.DD.YYYY (Puntos)"
    )
)

# Separamos los componentes por guion, barra o punto y los convertimos a enteros
# En regex, [-\/.] significa separar por '-', '/' o '.'
df_posiciones = df_ambiguos.withColumn(
    "partes", F.split(F.col("fecha"), r"[-\/\.]")
).withColumn(
    "pos_1", F.col("partes").getItem(0).cast("int")
).withColumn(
    "pos_2", F.col("partes").getItem(1).cast("int")
)

# Agrupamos y calculamos los máximos y banderas de control para ambas posiciones
df_posiciones.groupBy("formato_detectado") \
 .agg(
     F.max("pos_1").alias("Max_Posicion_1"),
     F.count(F.when(F.col("pos_1") > 12, 1)).alias("Casos_Pos1_>_12"),
     F.max("pos_2").alias("Max_Posicion_2"),
     F.count(F.when(F.col("pos_2") > 12, 1)).alias("Casos_Pos2_>_12")
 ).show(truncate=False)

##############################################################################
print(f"💡 MATRIZ DE DESEMPATE PARA REGISTROS ISO (¿AAAA-MM-DD o AAAA-DD-MM?):")

# Filtramos los registros clasificados como ISO (Guiones y Barras)
df_iso = df_formatos.filter(F.col("formato_detectado") == "YYYY-MM-DD (ISO / Estilo ISO)")

# Separamos por guion o barra. Index 0 = Año, Index 1 = Posición Media, Index 2 = Posición Final
df_iso_posiciones = df_iso.withColumn(
    "partes_iso", F.split(F.col("fecha"), r"[-\/]")
).withColumn(
    "pos_medio", F.col("partes_iso").getItem(1).cast("int")  # Debería ser el Mes (1-12)
).withColumn(
    "pos_final", F.col("partes_iso").getItem(2).cast("int")  # Debería ser el Día (1-31)
)

# Agrupamos para calcular los máximos y detectar si el medio supera el mes 12
df_iso_posiciones.agg(
    F.max("pos_medio").alias("Max_Posicion_Medio (Mes?)"),
    F.count(F.when(F.col("pos_medio") > 12, 1)).alias("Casos_Medio_>_12 (Alerta AAAA-DD-MM)"),
    F.max("pos_final").alias("Max_Posicion_Final (Día?)"),
    F.count(F.when(F.col("pos_final") > 12, 1)).alias("Casos_Final_>_12 (Valida AAAA-MM-DD)")
).show(truncate=False)

🔬 DIAGNÓSTICO MAESTRO: ANÁLISIS DE FORMATOS Y DESEMPATE CRUZADO EN 'fecha'
🔹 Frecuencia de formatos detectados en la columna:
+---------------------------------+-----+
|formato_detectado                |count|
+---------------------------------+-----+
|YYYY-MM-DD (ISO / Estilo ISO)    |236  |
|DD/MM/YYYY o MM/DD/YYYY (Barras) |92   |
|DD-MM-YYYY o MM-DD-YYYY (Guiones)|65   |
|DD.MM.YYYY o MM.DD.YYYY (Puntos) |36   |
+---------------------------------+-----+

💡 MATRIZ DE DESEMPATE ANALÍTICO (Posición 1 vs Posición 2):
+---------------------------------+--------------+---------------+--------------+---------------+
|formato_detectado                |Max_Posicion_1|Casos_Pos1_>_12|Max_Posicion_2|Casos_Pos2_>_12|
+---------------------------------+--------------+---------------+--------------+---------------+
|DD.MM.YYYY o MM.DD.YYYY (Puntos) |28            |19             |12            |0              |
|DD-MM-YYYY o MM-DD-YYYY (Guiones)|27            |31             |12            |0   

In [20]:
# 1. Limpieza de la columna fecha
print(f"\n🔹 Limpiando la columna fecha...")
print(SUB_LINE)

##############################################################################
#  APLICAR PIPELINE DE CONVERSIÓN CON COALESCE
# Se evalúan en cascada. El primero que logre parsear la fecha define el valor de la fila.
df_transacciones_clean = df_transacciones_clean.withColumn(
    "fecha_clean",
    F.coalesce(
        F.try_to_date(F.col("fecha"), "yyyy-MM-dd"),       # 1. Estándar ISO
        F.try_to_date(F.col("fecha"), "yyyy/MM/dd"),       # 1. Estándar ISO con barras
        F.try_to_date(F.col("fecha"), "dd/MM/yyyy"),       # 2. Latam con barras
        F.try_to_date(F.col("fecha"), "dd-MM-yyyy"),       # 3. Latam con guiones
        F.try_to_date(F.col("fecha"), "dd.MM.yyyy")          # 5. Latam con puntos
    )
)

##############################################################################
# MOSTRAR MUESTRA FINAL (Después - Todo unificado en yyyy-MM-dd)
print(f"\n🔹 Muestra de fechas NORMALIZADAS (Tipo DateType nativo de Spark):")
df_transacciones_clean.select("id_cliente", "fecha_clean").show(10, truncate=False)

# 4. AUDITORÍA DE CALIDAD: Asegurar que no generamos nulos nuevos por error de casteo
# Contamos cuántos nulos/vacíos había antes vs cuántos hay ahora
print(f"\n🔬 Control de calidad post-casteo:")
print(SUB_LINE)
total_nulos_final = df_transacciones_clean.filter(F.col("fecha_clean").isNull()).count()
print(f"✅ Conversión finalizada. Total de registros que quedaron como NULL: {total_nulos_final}")


🔹 Limpiando la columna fecha...
------------------------------------------------------------------------------------------

🔹 Muestra de fechas NORMALIZADAS (Tipo DateType nativo de Spark):
+----------+-----------+
|id_cliente|fecha_clean|
+----------+-----------+
|1013      |2025-03-04 |
|1007      |2024-02-16 |
|1046      |2024-09-17 |
|1041      |2024-08-22 |
|1015      |2024-11-09 |
|1033      |2024-08-26 |
|1006      |2024-03-05 |
|1030      |2024-12-05 |
|1045      |2024-02-16 |
|1038      |2024-07-16 |
+----------+-----------+
only showing top 10 rows

🔬 Control de calidad post-casteo:
------------------------------------------------------------------------------------------
✅ Conversión finalizada. Total de registros que quedaron como NULL: 0


#### Limpieza de `plazo_dias `

In [21]:
print(SEPARATOR)
print("🔬 AUDITORÍA ANALÍTICA: DETECCIÓN DE CARACTERES EN PLAZO")
print(SEPARATOR)

# 1. Creamos una columna temporal que extraiga CUALQUIER carácter que NO sea un número
# El regex [0-9] se elimina, dejando solo letras, espacios, símbolos, etc.
df_auditoria_plazo = df_transacciones_clean.withColumn(
    "caracteres_no_numericos",
    F.regexp_replace(F.col("plazo_dias"), r"[0-9]", "")
).withColumn(
    "tipo_contenido",
    F.when(F.col("plazo_dias").isNull() | (F.col("plazo_dias") == ""), F.lit("1. NULL o Vacío"))
     .when(F.trim(F.col("caracteres_no_numericos")) == "", F.lit("2. Número Puro (Solo dígitos)"))
     .otherwise(F.concat(F.lit("3. Contiene: '"), F.col("caracteres_no_numericos"), F.lit("'")))
)



# 2. Mostrar el consolidado general de qué hay en esa columna
print("📊 Resumen de tipos de datos encontrados en el plazo:")
df_auditoria_plazo.groupBy("tipo_contenido") \
    .count() \
    .sort("tipo_contenido") \
    .show(truncate=False)



# 3. Mostrar todos los casos que traen texto o símbolos
print("📋 Registros con caracteres no numéricos mezclados:")
df_auditoria_plazo.filter(F.col("tipo_contenido").contains("3. Contiene")) \
    .select("plazo_dias", "caracteres_no_numericos") \
    .distinct() \
    .show(truncate=False)

🔬 AUDITORÍA ANALÍTICA: DETECCIÓN DE CARACTERES EN PLAZO
📊 Resumen de tipos de datos encontrados en el plazo:
+-----------------------------+-----+
|tipo_contenido               |count|
+-----------------------------+-----+
|1. NULL o Vacío              |120  |
|2. Número Puro (Solo dígitos)|168  |
|3. Contiene: ' dias'         |45   |
|3. Contiene: '-'             |58   |
|3. Contiene: 'N/A'           |38   |
+-----------------------------+-----+

📋 Registros con caracteres no numéricos mezclados:
+----------+-----------------------+
|plazo_dias|caracteres_no_numericos|
+----------+-----------------------+
|N/A       |N/A                    |
|-         |-                      |
|365 dias  | dias                  |
+----------+-----------------------+



In [22]:
print(f"\n🔹 Limpiando y unificando la columna plazo_dias...")
print(SUB_LINE)

# 1. Aplicamos el pipeline de limpieza
df_transacciones_clean = df_transacciones_clean.withColumn(
    "plazo_dias_clean",
    F.when(
        F.col("plazo_dias").isNull() | 
        (F.trim(F.col("plazo_dias")) == "") | 
        (F.trim(F.col("plazo_dias")) == "-") | 
        (F.upper(F.trim(F.col("plazo_dias"))) == "N/A"),
        F.lit(None) # Todo lo vacío, guiones o N/A se estandariza a NULL nativo
    )
    .otherwise(
        # Eliminamos " dias" (con espacio) y pasamos a entero (no vimos puntos o comas en el string)
        # Usamos r"(?i) dias" para que sea insensible a mayúsculas/minúsculas por seguridad
        F.regexp_replace(F.col("plazo_dias"), r" dias", "").cast("int")
    )
)

# --- AUDITORÍA DE CALIDAD ---
print("📋 Todos los de plazos normalizados (Tipo Integer):")
df_transacciones_clean.filter(F.col("plazo_dias").isNotNull()) \
    .select("plazo_dias", "plazo_dias_clean") \
    .distinct() \
    .show(15, truncate=False)


🔹 Limpiando y unificando la columna plazo_dias...
------------------------------------------------------------------------------------------
📋 Todos los de plazos normalizados (Tipo Integer):
+----------+----------------+
|plazo_dias|plazo_dias_clean|
+----------+----------------+
|360       |360             |
|180       |180             |
|365 dias  |365             |
|0         |0               |
|90        |90              |
|-         |NULL            |
|N/A       |NULL            |
+----------+----------------+



#### Limpieza de `monto `

In [23]:
# --- 1. CONFIGURACIÓN DE FORMATO: TRIM Y NORMALIZACIÓN DE NULOS ---
print(f"\n🔹 Aplicando Trim y normalizando textos nulos en columna monto...")
print(SEPARATOR)

# Creamos la columna inicial con Trim
df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_clean", 
    F.trim(F.col("monto"))
)

# Convertimos las cadenas de texto basura o vacías en nulos reales (None)
df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_clean",
    F.when(
        F.col("monto_clean").isNull() | 
        F.upper(F.col("monto_clean")).isin("SIN DATO", "-", "NULL", "N/A", ""),
        F.lit(None).cast("string")
    ).otherwise(F.col("monto_clean"))
)

# Ahora removemos el guion inicial (^-) en los registros que sobrevivieron
df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_clean",
    F.when(
        F.col("monto_clean").isNotNull(),
        F.regexp_replace(F.col("monto_clean"), "^-", "")
    ).otherwise(F.col("monto_clean"))
)


####################################################################
# --- SCRIPT DE DIAGNÓSTICO: AISLANDO RESIDUOS EN MONTO ---
print(SEPARATOR)
print("🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'monto_clean'")
print(SEPARATOR)

# Filtramos primero para evaluar únicamente los registros con datos válidos (post-limpieza de nulos y guion)
df_validos = df_transacciones_clean.filter(F.col("monto_clean").isNotNull())

# Creamos la columna de residuo removiendo números y espacios en blanco
df_diagnostico_monto = df_validos.withColumn(
    "solo_ruido",
    F.regexp_replace(F.col("monto_clean"), r"[0-9.,]", "")
)

# Le damos una etiqueta visible a los vacíos (los que eran puramente números)
df_diagnostico_monto = df_diagnostico_monto.withColumn(
    "solo_ruido_visible",
    F.when(F.col("solo_ruido") == "", "[SÓLO NÚMEROS]")
     .otherwise(F.col("solo_ruido"))
)

# Agrupamos por los residuos encontrados para ver la tabla de frecuencias
print(f"🔹 Residuos consolidados en la columna monto:")
df_diagnostico_monto.groupBy("solo_ruido_visible") \
    .count() \
    .sort(F.desc("count")) \
    .show(30, truncate=False)


🔹 Aplicando Trim y normalizando textos nulos en columna monto...
🔬 DIAGNÓSTICO: CARACTERES NO NUMÉRICOS EN 'monto_clean'
🔹 Residuos consolidados en la columna monto:
+------------------+-----+
|solo_ruido_visible|count|
+------------------+-----+
|[SÓLO NÚMEROS]    |219  |
|$                 |149  |
| COP              |37   |
+------------------+-----+



In [24]:
# --- SCRIPT DE CONTROL: CRUCE DE NULOS MONTO VS MONEDA ---
print(SEPARATOR)
print("🔬 MATRIZ DE CONTROL: REGISTROS NULOS EN MONTO VS MONEDA")
print(SEPARATOR)

df_cruce_nulos = df_transacciones_clean.withColumn(
    "Estado_Monto",
    F.when(F.col("monto_clean").isNull(), "Monto_Nulo").otherwise("Monto_Con_Dato")
).withColumn(
    "Estado_Moneda",
    F.when(F.col("moneda_clean") == "NULL" , "Moneda_Nula").otherwise("Moneda_Con_Dato")
)

# Creamos la tabla cruzada de frecuencias
df_cruce_nulos.groupBy("Estado_Monto") \
    .pivot("Estado_Moneda") \
    .count() \
    .na.fill(0) \
    .show(truncate=False)

print(SEPARATOR)

#################################################################################
# --- SCRIPT DE INVESTIGACIÓN: DETECTANDO PISTAS DE MONEDA EN REGISTROS CRÍTICOS ---
print(SEPARATOR)
print("🔬 INSPECCIONANDO LOS 50 REGISTROS CRÍTICOS (MONTO SÍMBOLO Y MONEDA NULA)")
print(SEPARATOR)

# 1. Aislamos los 50 registros huérfanos usando las mismas condiciones del cruce
df_criticos = df_transacciones_clean.filter(
    F.col("monto_clean").isNotNull() & 
    (F.col("moneda_clean") == "NULL")
)

# 2. Creamos columnas booleanas rápidas para ver si el texto del monto nos da pistas
df_criticos_analisis = df_criticos.withColumn(
    "Contiene_Signo_Pesos", F.col("monto").contains("$")
).withColumn(
    "Contiene_Palabra_COP", F.col("monto").contains("COP")
)

# 3. Primero, lanzamos un conteo rápido para ver si el patrón es masivo
print("📊 Frecuencia de pistas encontradas en el texto de 'monto':")
df_criticos_analisis.groupBy("Contiene_Signo_Pesos", "Contiene_Palabra_COP") \
    .count() \
    .show(truncate=False)


################################################################################################
# --- REPARACIÓN DE MONEDA BASADA EN PISTAS DEL MONTO ---
print(f"\n🔹 Reparando valores nulos en 'moneda' usando patrones de 'monto'...")

# Definimos la condición para identificar una moneda inválida o ausente
moneda_es_nula = F.col("moneda_clean") == "NULL"
# Definimos las pistas visuales que encontramos en el campo original de monto
monto_tiene_pistas_cop = F.col("monto_clean").contains("$") | F.col("monto_clean").contains("COP")

# Contamos cuántos registros cumplen exactamente con la regla de rescate antes de aplicarla
cantidad_a_corregir = df_transacciones_clean.filter(moneda_es_nula & monto_tiene_pistas_cop).count()

# --- APLICACIÓN DE LA CORRECCIÓN ---
df_transacciones_clean = df_transacciones_clean.withColumn(
    "moneda_clean",
    F.when(moneda_es_nula & monto_tiene_pistas_cop, F.lit("COP"))
     .otherwise(F.col("moneda_clean"))
)

print(f"  • ✅ Registros a los que se les asignó 'COP': {cantidad_a_corregir}")

🔬 MATRIZ DE CONTROL: REGISTROS NULOS EN MONTO VS MONEDA
+--------------+---------------+-----------+
|Estado_Monto  |Moneda_Con_Dato|Moneda_Nula|
+--------------+---------------+-----------+
|Monto_Nulo    |21             |3          |
|Monto_Con_Dato|355            |50         |
+--------------+---------------+-----------+

🔬 INSPECCIONANDO LOS 50 REGISTROS CRÍTICOS (MONTO SÍMBOLO Y MONEDA NULA)
📊 Frecuencia de pistas encontradas en el texto de 'monto':
+--------------------+--------------------+-----+
|Contiene_Signo_Pesos|Contiene_Palabra_COP|count|
+--------------------+--------------------+-----+
|true                |false               |24   |
|false               |false               |23   |
|false               |true                |3    |
+--------------------+--------------------+-----+


🔹 Reparando valores nulos en 'moneda' usando patrones de 'monto'...
  • ✅ Registros a los que se les asignó 'COP': 27


In [25]:
# --- LIMPIEZA DE CARACTERES DE LA COLUMNA MONTO_CLEAN ---
print(f"\n🔹 Eliminando $, COP y espacios intermedios en 'monto_clean'...")

df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_clean",
    # Reemplazamos las expresiones conocidas: el '$', la palabra 'COP' y los espacios '\\s' por vacío
    F.regexp_replace(F.col("monto_clean"), r"[\$|COP|\s]", "")
)

# --- VERIFICACIÓN DE CONTROL ---
# Filtramos para verificar qué caracteres sobrevivieron (deberían ser SOLO números, puntos y comas)
df_verificacion_monto = df_transacciones_clean.filter(F.col("monto_clean").isNotNull()) \
    .withColumn("solo_ruido", F.regexp_replace(F.col("monto_clean"), r"[0-9]", ""))

print("📋 Muestra de los residuos sobrevivientes en 'monto_clean' (Puntos y Comas esperados):")
df_verificacion_monto.groupBy("solo_ruido") \
    .count() \
    .sort(F.desc("count")) \
    .show(20, truncate=False)


🔹 Eliminando $, COP y espacios intermedios en 'monto_clean'...
📋 Muestra de los residuos sobrevivientes en 'monto_clean' (Puntos y Comas esperados):
+----------+-----+
|solo_ruido|count|
+----------+-----+
|..        |280  |
|..,       |74   |
|,.        |38   |
|.         |10   |
|.,        |3    |
+----------+-----+



In [26]:
# --- NORMALIZACIÓN FINAL DE SEPARADORES EN MONTO ---
print(f"\n🔹 Estandarizando puntos y comas en 'monto_clean'...")

# Creamos variables de posición para hacer las condiciones legibles
pos_punto = F.locate(".", F.col("monto_clean"))
pos_coma = F.locate(",", F.col("monto_clean"))

df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_clean",
    F.when(
        # CASO 1: Solo hay puntos (no hay comas)
        (pos_punto > 0) & (pos_coma == 0),
        F.regexp_replace(F.col("monto_clean"), r"\.", "")
    )
    .when(
        # CASO 2: Hay coma y punto, y la coma está ANTES que el punto (ej. 1,250.50)
        (pos_punto > 0) & (pos_coma > 0) & (pos_coma < pos_punto),
        F.regexp_replace(F.col("monto_clean"), ",", "")
    )
    .when(
        # CASO 3: Hay punto y coma, y el punto está ANTES que la coma (ej. 1.250,50)
        (pos_punto > 0) & (pos_coma > 0) & (pos_punto < pos_coma),
        # Primero borramos los puntos, y al resultado le cambiamos la coma por punto
        F.regexp_replace(F.regexp_replace(F.col("monto_clean"), r"\.", ""), ",", ".")
    )
    # Si no cumple ninguna (ya es un número limpio), se deja igual
    .otherwise(F.col("monto_clean"))
)

#####################################################################
# --- CASTEO FINAL A DECIMAL ---
df_transacciones_clean = df_transacciones_clean.withColumn(
    "monto_final", 
    F.col("monto_clean").cast("decimal(18,2)")
)

# --- CONTROL DE CALIDAD ---
print("🔬 MUESTRA DE VALIDACIÓN DE LOS MONTOS CORREGIDOS:")
df_transacciones_clean.filter(F.col("monto").rlike(r"[.,]")) \
    .select("monto", "monto_clean", "monto_final") \
    .distinct() \
    .show(20, truncate=False)

# Conteo de posibles fallas en el casteo
fallas_casteo = df_transacciones_clean.filter(F.col("monto_clean").isNotNull() & F.col("monto_final").isNull()).count()
print(f"⚠️ Registros que fallaron al convertirse a número: {fallas_casteo}")


🔹 Estandarizando puntos y comas en 'monto_clean'...
🔬 MUESTRA DE VALIDACIÓN DE LOS MONTOS CORREGIDOS:
+--------------+-----------+-----------+
|monto         |monto_clean|monto_final|
+--------------+-----------+-----------+
|$77.374.240   |77374240   |77374240.00|
|23.191.663,50 |23191663.50|23191663.50|
|$62.623.188   |62623188   |62623188.00|
|731.12        |73112      |73112.00   |
|-10.659.303   |10659303   |10659303.00|
| 29.206.934   |29206934   |29206934.00|
|$7.136.176    |7136176    |7136176.00 |
|-3.877.402    |3877402    |3877402.00 |
| 51.132.338   |51132338   |51132338.00|
|$59.872.735   |59872735   |59872735.00|
|75.892.796 COP|75892796   |75892796.00|
|$857.572      |857572     |857572.00  |
|45.815.428,00 |45815428.00|45815428.00|
|$33.668.051   |33668051   |33668051.00|
|2.839.929     |2839929    |2839929.00 |
|3.689.661,25  |3689661.25 |3689661.25 |
| 11.604.150   |11604150   |11604150.00|
|$14.817.229   |14817229   |14817229.00|
|2.388.293,00  |2388293.00 |2388293.

#### Limpieza de `tasa_interes  `

In [27]:
# --- CONFIGURACIÓN DE FORMATO ---
print(f"\n🔹 Limpieza inicial de la columna tasa_interes...")
print(SEPARATOR)

# 1. Estandarizamos espacios y convertimos N/A y vacíos a NULL reales
df_transacciones_clean = df_transacciones_clean.withColumn(
    "tasa_interes_clean",
    F.when(
        F.col("tasa_interes").isNull() | 
        (F.trim(F.col("tasa_interes")) == "") | 
        (F.upper(F.trim(F.col("tasa_interes"))) == "N/A"),
        F.lit(None)
    )
    .otherwise(F.trim(F.col("tasa_interes"))) # Conservamos el texto limpio de los datos válidos
)

# Contamos cuántos registros logramos estandarizar a NULL
nulos_normalizados = df_transacciones_clean.filter(
    F.col("tasa_interes_clean").isNull() & 
    F.col("tasa_interes").isNotNull()
).count()

print(f"✅ Fase inicial completada. Cadenas 'N/A' y vacíos convertidos a NULL: {nulos_normalizados}")

#############################################################################
# --- CONTEO TOTAL DE NULOS EN TASA_INTERES_CLEAN ---
reporte_nulos = df_transacciones_clean.agg(
    F.count("*").alias("Total_Registros"),
    F.sum(F.when(F.col("tasa_interes_clean").isNull(), 1).otherwise(0)).alias("Total_Nulos_Finales")
).withColumn(
    "Porcentaje_Nulos", 
    F.round((F.col("Total_Nulos_Finales") / F.col("Total_Registros")) * 100, 2)
)

print("\n📊 BALANCE FINAL DE NULOS EN LA COLUMNA:")
reporte_nulos.show()


🔹 Limpieza inicial de la columna tasa_interes...
✅ Fase inicial completada. Cadenas 'N/A' y vacíos convertidos a NULL: 90

📊 BALANCE FINAL DE NULOS EN LA COLUMNA:
+---------------+-------------------+----------------+
|Total_Registros|Total_Nulos_Finales|Porcentaje_Nulos|
+---------------+-------------------+----------------+
|            429|                174|           40.56|
+---------------+-------------------+----------------+



In [28]:
# --- SCRIPT DE AUDITORÍA: FORMATOS DE TASAS POR TIPO DE PRODUCTO ---
print(SEPARATOR)
print("🔬 MATRIZ DE FORMATOS EN TASA_INTERES_CLEAN POR tipo_producto_clean")
print(SEPARATOR)

df_matriz_formatos = df_transacciones_clean.groupBy("tipo_producto_clean") \
    .agg(
        F.count("*").alias("Total_Registros"),
        # Conteo de nulos actuales en la columna limpia
        F.sum(F.when(F.col("tasa_interes_clean").isNull(), 1).otherwise(0)).alias("Nulos"),
        # Conteo por presencia del signo %
        F.sum(F.when(F.col("tasa_interes_clean").contains("%"), 1).otherwise(0)).alias("Con_Porcentaje"),
        # Conteo por tipo de separador decimal
        F.sum(F.when(F.col("tasa_interes_clean").contains(","), 1).otherwise(0)).alias("Con_Coma"),
        F.sum(F.when(F.col("tasa_interes_clean").contains("."), 1).otherwise(0)).alias("Con_Punto"),
        # Casos mixtos o enteros puros (sin puntos ni comas, ej: "12")
        F.sum(
            F.when(
                F.col("tasa_interes_clean").isNotNull() & 
                ~F.col("tasa_interes_clean").contains(",") & 
                ~F.col("tasa_interes_clean").contains(".") &
                ~F.col("tasa_interes_clean").contains("%"), 1
            ).otherwise(0)
        ).alias("Enteros")
    ) \
    .sort(F.desc("Total_Registros"))

# Mostramos la matriz completa
df_matriz_formatos.show(truncate=False)

#######################################################################################
# --- SCRIPT DE AUDITORÍA: EVALUACIÓN DE CEROS EN CUENTAS ---
print(SEPARATOR)
print("🔬 ANALIZANDO EL VALOR DE LAS TASAS EN CUENTAS (AHORROS Y CORRIENTE)")
print(SEPARATOR)

# Filtramos únicamente para los productos de tipo cuenta que tengan datos (no nulos)
df_cuentas = df_transacciones_clean.filter(
    F.col("tipo_producto_clean").isin("Cuenta Ahorros", "Cuenta Corriente") & 
    F.col("tasa_interes_clean").isNotNull()
)

# Agrupamos por producto y por el valor exacto del string para ver qué números hay
print("📊 Distribución de valores reales encontrados en las tasas de las cuentas:")
df_cuentas.groupBy("tipo_producto_clean", "tasa_interes_clean") \
    .agg(F.count("*").alias("Cantidad_Registros")) \
    .sort("tipo_producto_clean", F.desc("Cantidad_Registros")) \
    .show(truncate=False)

# Conteo directo de cuántos son exactamente "0" 
conteo_ceros = df_cuentas.filter(F.col("tasa_interes_clean") == "0").count()
total_con_datos = df_cuentas.count()

print(SEPARATOR)
print(f"🎯 Resumen de Cuentas con Datos:")
print(f"  • Total de cuentas con valor asignado (no nulos): {total_con_datos}")
print(f"  • ¿Cuántos de esos valores son exactamente CERO?: {conteo_ceros}")
print(f"  • ¿Cuántos tienen un valor diferente de cero?: {total_con_datos - conteo_ceros}")

print(SEPARATOR)

🔬 MATRIZ DE FORMATOS EN TASA_INTERES_CLEAN POR tipo_producto_clean
+-------------------+---------------+-----+--------------+--------+---------+-------+
|tipo_producto_clean|Total_Registros|Nulos|Con_Porcentaje|Con_Coma|Con_Punto|Enteros|
+-------------------+---------------+-----+--------------+--------+---------+-------+
|Cuenta Ahorros     |147            |100  |0             |0       |0        |47     |
|CDT                |146            |20   |64            |65      |61       |0      |
|Credito            |76             |12   |42            |38      |26       |0      |
|Cuenta Corriente   |60             |42   |0             |0       |0        |18     |
+-------------------+---------------+-----+--------------+--------+---------+-------+

🔬 ANALIZANDO EL VALOR DE LAS TASAS EN CUENTAS (AHORROS Y CORRIENTE)
📊 Distribución de valores reales encontrados en las tasas de las cuentas:
+-------------------+------------------+------------------+
|tipo_producto_clean|tasa_interes_clean|Ca

In [29]:
# --- CONFIGURACIÓN DE FORMATO de comas a puntos y eliminación de porcentajes ---
print(f"\n🔹 Normalizando separadores decimales y eliminando signos de porcentaje...")

# Encadenamos los reemplazos: primero comas por puntos, y luego quitamos el '%'
df_transacciones_clean = df_transacciones_clean.withColumn(
    "tasa_interes_clean",
    F.regexp_replace(
        F.regexp_replace(F.col("tasa_interes_clean"), ",", "."), 
        "%", ""
    ).cast("decimal(10,6)")
)

##########################################################################
# --- SCRIPT EXPLORATORIO: RESUMEN DE TASAS NUMÉRICAS ---
print(SEPARATOR)
print("🔬 ANÁLISIS EXPLORATORIO DE TASAS NUMÉRICAS (PRE-AJUSTE DE ESCALA)")
print(SEPARATOR)

df_exploratorio_tasas = df_transacciones_clean \
    .filter(F.col("tipo_producto_clean").isin("Credito", "CDT")) \
    .groupBy("tipo_producto_clean") \
    .agg(
        F.count("*").alias("Total_Registros"),
        # Contamos cuántos de estos registros tienen datos válidos (no nulos)
        F.sum(F.when(F.col("tasa_interes_clean").isNotNull(), 1).otherwise(0)).alias("Registros_Con_Dato"),
        # Contamos cuántos nulos quedan en estos productos
        F.sum(F.when(F.col("tasa_interes_clean").isNull(), 1).otherwise(0)).alias("Registros_Nulos"),
        # Métricas directas sobre la columna decimal
        F.round(F.min("tasa_interes_clean"), 4).alias("Min_Tasa"),
        F.round(F.avg("tasa_interes_clean"), 4).alias("Prom_Tasa"),
        F.round(F.max("tasa_interes_clean"), 4).alias("Max_Tasa")
    )

# Mostramos el reporte descriptivo simple
df_exploratorio_tasas.show(truncate=False)


###########################################################################
# --- SCRIPT DE AUDITORÍA: FRONTERA DE ESCALAS (BASE 1 VS BASE 100) ---
print(SEPARATOR)
print("🔬 ANALIZANDO FRONTERA DE ESCALAS (BASE 1 VS BASE 100) EN EL UNIVERSO DE TASAS")
print(SEPARATOR)

# Filtramos los registros que tienen datos válidos y corresponden a Crédito o CDT
df_analisis_frontera = df_transacciones_clean.filter(
    F.col("tasa_interes_clean").isNotNull() &
    F.col("tipo_producto_clean").isin("Credito", "CDT")
)

# Calculamos las métricas de frontera operando directamente sobre la columna decimal
df_frontera = df_analisis_frontera.agg(
    # Grupo de los menores o iguales a 1 (Mapeados en tanto por uno: ej. 0.082)
    F.count(F.when(F.col("tasa_interes_clean") <= 1, 1)).alias("Cantidad_Menores_o_Igual_1"),
    F.round(F.min(F.when(F.col("tasa_interes_clean") <= 1, F.col("tasa_interes_clean"))), 4).alias("Minimo_Menor_1"),
    F.round(F.max(F.when(F.col("tasa_interes_clean") <= 1, F.col("tasa_interes_clean"))), 4).alias("Maximo_Menor_1"),
    
    # Grupo de los mayores a 1 (Mapeados en tanto por ciento: ej. 13.14)
    F.count(F.when(F.col("tasa_interes_clean") > 1, 1)).alias("Cantidad_Mayores_1"),
    F.round(F.min(F.when(F.col("tasa_interes_clean") > 1, F.col("tasa_interes_clean"))), 4).alias("Minimo_Mayor_1"),
    F.round(F.max(F.when(F.col("tasa_interes_clean") > 1, F.col("tasa_interes_clean"))), 4).alias("Maximo_Mayor_1")
)

df_frontera.show()
print(SEPARATOR)


🔹 Normalizando separadores decimales y eliminando signos de porcentaje...
🔬 ANÁLISIS EXPLORATORIO DE TASAS NUMÉRICAS (PRE-AJUSTE DE ESCALA)
+-------------------+---------------+------------------+---------------+--------+---------+--------+
|tipo_producto_clean|Total_Registros|Registros_Con_Dato|Registros_Nulos|Min_Tasa|Prom_Tasa|Max_Tasa|
+-------------------+---------------+------------------+---------------+--------+---------+--------+
|CDT                |146            |126               |20             |0.0810  |8.6745   |14.0000 |
|Credito            |76             |64                |12             |0.0820  |8.8372   |13.9000 |
+-------------------+---------------+------------------+---------------+--------+---------+--------+

🔬 ANALIZANDO FRONTERA DE ESCALAS (BASE 1 VS BASE 100) EN EL UNIVERSO DE TASAS
+--------------------------+--------------+--------------+------------------+--------------+--------------+
|Cantidad_Menores_o_Igual_1|Minimo_Menor_1|Maximo_Menor_1|Cantidad

In [30]:
# --- CONFIGURACIÓN DE FORMATO ---
print(f"\n🔹 Unificando escalas y casteando tasa_interes a flotante...")
print(SEPARATOR)

# Aplicamos la regla de negocio de escalas (Multiplicar por 100 si es <= 1)
df_transacciones_clean = df_transacciones_clean.withColumn(
    "tasa_interes_clean",
    F.when(
        F.col("tasa_interes_clean").isNotNull() & 
        (F.col("tasa_interes_clean") > 0) & 
        (F.col("tasa_interes_clean") <= 1),
        F.round(F.col("tasa_interes_clean") * 100, 4)
    )
    .otherwise(F.col("tasa_interes_clean")) # Mantiene los > 1, los 0 y los NULL
)

# --- AUDITORÍA DE CONTROL FINAL ---
print("📋 Muestra final del comportamiento de tasas por producto:")
df_transacciones_clean.groupBy("tipo_producto_clean") \
    .agg(
        F.count("*").alias("Total"),
        F.sum(F.when(F.col("tasa_interes_clean").isNull(), 1).otherwise(0)).alias("Nulos"),
        F.min("tasa_interes_clean").alias("Min"),
        F.round(F.avg("tasa_interes_clean"), 2).alias("Promedio"),
        F.max("tasa_interes_clean").alias("Max")
    ).show(truncate=False)

print("✅ ¡Columna tasa_interes completamente normalizada y unificada!")


🔹 Unificando escalas y casteando tasa_interes a flotante...
📋 Muestra final del comportamiento de tasas por producto:
+-------------------+-----+-----+--------+--------+---------+
|tipo_producto_clean|Total|Nulos|Min     |Promedio|Max      |
+-------------------+-----+-----+--------+--------+---------+
|CDT                |146  |20   |8.100000|10.91   |14.000000|
|Credito            |76   |12   |8.000000|11.07   |13.900000|
|Cuenta Ahorros     |147  |100  |0.000000|0.00    |0.000000 |
|Cuenta Corriente   |60   |42   |0.000000|0.00    |0.000000 |
+-------------------+-----+-----+--------+--------+---------+

✅ ¡Columna tasa_interes completamente normalizada y unificada!


#### Limpieza de `id_cliente `

In [31]:
print(f"\n🔹 Limpiando y validando la columna id_cliente en transacciones...")

# Aplicamos trim y validamos que sea un número entero puro directamente sobre la columna
df_transacciones_clean = df_transacciones_clean.withColumn(
    "id_cliente_clean",
    F.when(
        F.col("id_cliente").isNull() | (F.trim(F.col("id_cliente")) == ""), 
        F.lit(None)
    )
    .when(
        # Validamos que consista solo de dígitos
        F.trim(F.col("id_cliente")).rlike(r"^\d+$"),
        F.trim(F.col("id_cliente")).cast("long") # Tipo 'long' para el cruzamiento futuro
    )
    .otherwise(F.lit(None)) # Cualquier ID con letras o caracteres especiales pasa a NULL
)

# --- AUDITORÍA DE CALIDAD ---
print("📋 Muestra de IDs de cliente en transacciones:")
df_transacciones_clean.select("id_cliente", "id_cliente_clean").show(10, truncate=False)

# Contamos cuántos registros de transacciones tenían IDs de cliente corruptos, excluyendo vacíos
transacciones_clientes_corruptos = df_transacciones_clean.filter(
    F.col("id_cliente_clean").isNull() & 
    F.col("id_cliente").isNotNull() & 
    (F.col("id_cliente") != "")
).count()

print(f"✅ Validación finalizada. Transacciones con id_cliente corrupto enviadas a NULL: {transacciones_clientes_corruptos}")


🔹 Limpiando y validando la columna id_cliente en transacciones...
📋 Muestra de IDs de cliente en transacciones:
+----------+----------------+
|id_cliente|id_cliente_clean|
+----------+----------------+
|1013      |1013            |
|1007      |1007            |
|1046      |1046            |
|1041      |1041            |
|1015      |1015            |
|1033      |1033            |
|1006      |1006            |
|1030      |1030            |
|1045      |1045            |
|1038      |1038            |
+----------+----------------+
only showing top 10 rows
✅ Validación finalizada. Transacciones con id_cliente corrupto enviadas a NULL: 0


#### Limpieza de `id_transaccion` 

In [32]:
print(f"\n🔹 Limpiando y validando la columna id_transaccion...")
print(SUB_LINE)

# Aplicamos trim y validamos que sea un número entero puro directamente sobre la columna
df_transacciones_clean = df_transacciones_clean.withColumn(
    "id_transaccion_clean",
    F.when(
        F.col("id_transaccion").isNull() | (F.trim(F.col("id_transaccion")) == ""), 
        F.lit(None)
    )
    .when(
        # Expresión regular: ^ (inicio), \d+ (uno o más dígitos), $ (fin)
        F.trim(F.col("id_transaccion")).rlike(r"^\d+$"),
        F.trim(F.col("id_transaccion")).cast("long") # 'long' para evitar desbordamientos de números grandes
    )
    .otherwise(F.lit(None)) # Si contiene letras o símbolos extraños, se va a NULL
)

# --- AUDITORÍA DE CALIDAD ---
print("📋 Muestra de IDs de transacción procesados:")
df_transacciones_clean.select("id_transaccion", "id_transaccion_clean").show(10, truncate=False)

# Contamos si había IDs con basura que pasaron a ser NULL
ids_corruptos = df_transacciones_clean.filter(
    F.col("id_transaccion_clean").isNull() & 
    F.col("id_transaccion").isNotNull()
).count()

print(f"✅ Validación finalizada. Registros corruptos (con letras/símbolos) enviados a NULL: {ids_corruptos}")


🔹 Limpiando y validando la columna id_transaccion...
------------------------------------------------------------------------------------------
📋 Muestra de IDs de transacción procesados:
+--------------+--------------------+
|id_transaccion|id_transaccion_clean|
+--------------+--------------------+
|500058        |500058              |
|500240        |500240              |
|500262        |500262              |
|500041        |500041              |
|500073        |500073              |
|500380        |500380              |
|500025        |500025              |
|500135        |500135              |
|500020        |500020              |
|500207        |500207              |
+--------------+--------------------+
only showing top 10 rows
✅ Validación finalizada. Registros corruptos (con letras/símbolos) enviados a NULL: 0


#### Consolidación de tabla transacciones

In [33]:
print(SEPARATOR)
print("✨ DEPURACIÓN FINAL: CONSTRUYENDO EL DATAFRAME TRANSACCIONES LIMPIO")
print(SEPARATOR)

# Creamos el DataFrame definitivo seleccionando y renombrando en un solo viaje
df_transacciones_final = df_transacciones_clean.select(
    F.col("id_transaccion_clean").alias("id_transaccion"),
    F.col("fecha_clean").alias("fecha"),
    F.col("id_cliente_clean").alias("id_cliente"),
    F.col("tipo_producto_clean").alias("tipo_producto"),
    F.col("monto_final").alias("monto"),               # Tipo Decimal(18,2)
    F.col("moneda_clean").alias("moneda"),
    F.col("tasa_interes_clean").alias("tasa_interes"),   # Tipo Decimal(10,6)
    F.col("plazo_dias_clean").alias("plazo_dias"),       # Tipo Integer
    F.col("canal_clean").alias("canal"),
    F.col("estado_clean").alias("estado"),
    F.col("sucursal_clean").alias("sucursal")
)

# Mostramos el nuevo esquema limpio
print("📋 Esquema final de la tabla 'transacciones_final':")
df_transacciones_final.printSchema()

print(f"🔹 Muestra de los datos listos:")
df_transacciones_final.show(5, truncate=False)
print(SEPARATOR)

✨ DEPURACIÓN FINAL: CONSTRUYENDO EL DATAFRAME TRANSACCIONES LIMPIO
📋 Esquema final de la tabla 'transacciones_final':
root
 |-- id_transaccion: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- tipo_producto: string (nullable = false)
 |-- monto: decimal(18,2) (nullable = true)
 |-- moneda: string (nullable = false)
 |-- tasa_interes: decimal(15,6) (nullable = true)
 |-- plazo_dias: integer (nullable = true)
 |-- canal: string (nullable = false)
 |-- estado: string (nullable = false)
 |-- sucursal: string (nullable = true)

🔹 Muestra de los datos listos:
+--------------+----------+----------+--------------+-----------+------+------------+----------+------+---------+------------+
|id_transaccion|fecha     |id_cliente|tipo_producto |monto      |moneda|tasa_interes|plazo_dias|canal |estado   |sucursal    |
+--------------+----------+----------+--------------+-----------+------+------------+----------+------+---------+------------+
|5000

---

## 🔄 Paso 3: Cruce de Fuentes

Se realiza un corto análisis de duplicados para proceder a la unión entre los datos limpios

### Registros Duplicados en Clientes

In [34]:
print("==========================================================================")
print("🔍 AUDITORÍA DE DUPLICADOS EN LA FILA COMPLETA (TABLA CLIENTES)")
print("==========================================================================")

# 1. Contar el total de registros actuales en el DataFrame de clientes
total_filas = df_clientes_final.count()

# 2. Aplicar distinct() para obtener las filas únicas considerando TODAS las columnas
df_clientes_unicos = df_clientes_final.distinct()
total_filas_unicas = df_clientes_unicos.count()

# 3. Calcular la cantidad de filas que son espejos idénticos
filas_duplicadas_exactas = total_filas - total_filas_unicas

print(f"-> Total de filas evaluadas: {total_filas}")
print(f"-> Total de filas únicas (con distinct): {total_filas_unicas}")
print(f"-> Cantidad de filas duplicadas exactas: {filas_duplicadas_exactas}")

# 4. Visualizar cuáles son esas filas que están repetidas por completo
if filas_duplicadas_exactas > 0:
    print("\n⚠️ Mostrando filas que tienen duplicados idénticos en el dataset:")
    # Agrupamos por todas las columnas para identificar cuáles aparecen más de una vez
    df_clientes_final.groupBy(df_clientes_final.columns) \
                     .count() \
                     .filter(F.col("count") > 1) \
                     .show(truncate=False)
else:
    print("\n✅ ¡Perfecto! No existen filas idénticas duplicadas en su totalidad.")

🔍 AUDITORÍA DE DUPLICADOS EN LA FILA COMPLETA (TABLA CLIENTES)
-> Total de filas evaluadas: 52
-> Total de filas únicas (con distinct): 50
-> Cantidad de filas duplicadas exactas: 2

⚠️ Mostrando filas que tienen duplicados idénticos en el dataset:
+----------+---------------+--------------+----------------+------------------------------------------+--------+--------+----------+------+------------------+-----+
|id_cliente|nombre         |tipo_documento|numero_documento|contacto                                  |ciudad  |segmento|fecha_alta|activo|productos         |count|
+----------+---------------+--------------+----------------+------------------------------------------+--------+--------+----------+------+------------------+-----+
|1004      |Isabella Diaz  |CC            |342601915       |{isabella.diaz@empresa.com.co, 3138849654}|Medellin|Premium |2022-01-07|true  |[Cuenta Ahorros]  |2    |
|1011      |Santiago Vargas|NIT           |339147326       |{NULL, 6025607689}             

In [35]:
print("==========================================================================")
print("🔍 VERIFICACIÓN DE DUPLICADOS POR LLAVE PRIMARIA (id_cliente)")
print("==========================================================================")

# 1. Aplicamos distinct() para remover los duplicados de fila completa encontrada
df_clientes_sin_duplicados_completos = df_clientes_final.distinct()

# 2. Contamos IDs totales vs IDs únicos
total_ids = df_clientes_sin_duplicados_completos.count()
unicos_ids = df_clientes_sin_duplicados_completos.select("id_cliente").distinct().count()
diferencia_ids = total_ids - unicos_ids

print(f"-> Total de registros (sin duplicados completos): {total_ids}")
print(f"-> Total de 'id_cliente' únicos: {unicos_ids}")
print(f"-> Registros con IDs repetidos detectados: {diferencia_ids}")

df_clientes_preparado = df_clientes_sin_duplicados_completos

🔍 VERIFICACIÓN DE DUPLICADOS POR LLAVE PRIMARIA (id_cliente)
-> Total de registros (sin duplicados completos): 50
-> Total de 'id_cliente' únicos: 50
-> Registros con IDs repetidos detectados: 0


### Registros Duplicados en Transacciones

In [36]:
print("==========================================================================")
print("🔍 AUDITORÍA DE DUPLICADOS EN LA FILA COMPLETA (TABLA TRANSACCIONES)")
print("==========================================================================")

# 1. Contar el total de registros en el DataFrame de transacciones tras la limpieza
total_tx = df_transacciones_final.count()

# 2. Aplicar distinct() para obtener los movimientos únicos considerando TODAS las columnas
df_transacciones_unicas = df_transacciones_final.distinct()
total_tx_unicas = df_transacciones_unicas.count()

# 3. Calcular la cantidad de filas que son duplicados idénticos
tx_duplicadas_exactas = total_tx - total_tx_unicas

print(f"-> Total de registros evaluados: {total_tx}")
print(f"-> Total de registros únicos (con distinct): {total_tx_unicas}")
print(f"-> Cantidad de filas duplicadas exactas: {tx_duplicadas_exactas}")

# 4. Visualizar una muestra de las filas que están repetidas por completo
if tx_duplicadas_exactas > 0:
    print("\n⚠️ Mostrando una muestra de filas con duplicados idénticos en transacciones:")
    df_transacciones_final.groupBy(df_transacciones_final.columns) \
                          .count() \
                          .filter(F.col("count") > 1) \
                          .show(5, truncate=False)
else:
    print("\n✅ ¡Excelente! No existen registros idénticos duplicados en su totalidad en esta tabla.")

🔍 AUDITORÍA DE DUPLICADOS EN LA FILA COMPLETA (TABLA TRANSACCIONES)
-> Total de registros evaluados: 429
-> Total de registros únicos (con distinct): 423
-> Cantidad de filas duplicadas exactas: 6

⚠️ Mostrando una muestra de filas con duplicados idénticos en transacciones:
+--------------+----------+----------+--------------+-----------+------+------------+----------+-------+---------+------------+-----+
|id_transaccion|fecha     |id_cliente|tipo_producto |monto      |moneda|tasa_interes|plazo_dias|canal  |estado   |sucursal    |count|
+--------------+----------+----------+--------------+-----------+------+------------+----------+-------+---------+------------+-----+
|500260        |2024-08-03|1006      |CDT           |52105212.50|COP   |10.200000   |NULL      |Web    |Rechazada|Bucaramanga |2    |
|500148        |2024-03-01|1033      |Cuenta Ahorros|NULL       |COP   |0.000000    |NULL      |App    |Pendiente|Bogota      |2    |
|500123        |2024-11-20|7777      |Cuenta Ahorros|19

In [37]:
print("==========================================================================")
print("🔍 VERIFICACIÓN DE DUPLICADOS POR LLAVE PRIMARIA (id_transaccion)")
print("==========================================================================")

# 1. Eliminamos los 6 duplicados exactos de fila completa mediante distinct()
df_transacciones_sin_duplicados_completos = df_transacciones_final.distinct()

# 2. Contamos IDs de transacción totales vs únicos
total_tx_ids = df_transacciones_sin_duplicados_completos.count()
unicos_tx_ids = df_transacciones_sin_duplicados_completos.select("id_transaccion").distinct().count()
diferencia_tx_ids = total_tx_ids - unicos_tx_ids

print(f"-> Total de registros (sin duplicados completos): {total_tx_ids}")
print(f"-> Total de 'id_transaccion' únicos: {unicos_tx_ids}")
print(f"-> Registros con IDs de transacción repetidos detectados: {diferencia_tx_ids}")

df_transacciones_preparado = df_transacciones_sin_duplicados_completos

🔍 VERIFICACIÓN DE DUPLICADOS POR LLAVE PRIMARIA (id_transaccion)
-> Total de registros (sin duplicados completos): 423
-> Total de 'id_transaccion' únicos: 423
-> Registros con IDs de transacción repetidos detectados: 0


### Unión de los Datos

In [38]:
print("==========================================================================")
print("🚀 EJECUTANDO CRUCE DE TABLAS (LEFT JOIN)")
print("==========================================================================")

# 1. Ejecutar el Left Join usando 'id_cliente' como llave de cruce
# Usamos df_transacciones_sin_duplicados_completos como base (tabla izquierda)
df_consolidado_raw = df_transacciones_preparado.join(
    df_clientes_preparado, # Tabla derecha
    on="id_cliente",
    how="left"
)

print(f"-> Total de transacciones consolidadas: {df_consolidado_raw.count()}")

🚀 EJECUTANDO CRUCE DE TABLAS (LEFT JOIN)
-> Total de transacciones consolidadas: 423


---

## ⚠️ 4. Casos Problemáticos

Se evalúa el tratameinto de casos particulares que han de ser vistos más a detalle

### Transacciones Huerfanas

In [39]:
# 1. Métrica A: Nombres ausentes o vacíos en el catálogo preparado
nombres_ausentes_cat = df_clientes_preparado.filter(
    (F.col("nombre").isNull()) | (F.trim(F.col("nombre")) == "")
).count()

# 2. Métrica B: Clientes únicos en transacciones que NO existen en el catálogo preparado
# Extraemos los clientes del consolidado donde los datos del catálogo se fueron a null
clientes_no_registrados_unicos = df_consolidado_raw.filter(
    F.col("nombre").isNull()
).select("id_cliente").distinct().count()

# 3. Métrica C: Volumen de transacciones realizadas por esos clientes no registrados
transacciones_huerfanas_totales = df_consolidado_raw.filter(
    F.col("nombre").isNull()
).select("id_transaccion").count()


import pandas as pd
# Creamos el diccionario con los datos locales que ya calculamos
data_local = {
    "Nombres_Ausentes_en_Clientes": [nombres_ausentes_cat],
    "Clientes_No_Registrados_en_Consolidado": [clientes_no_registrados_unicos],
    "Total_Transacciones_Huerfanas": [transacciones_huerfanas_totales]
}

# Creamos el dataframe de Pandas
df_pandas_control = pd.DataFrame(data_local)
print(df_pandas_control.to_string(index=False))


# Aplicamos la regla: si el nombre es nulo tras el join, se marca como "No identificado"
df_consolidado_raw = df_consolidado_raw.withColumn(
    "nombre",
    F.when(F.col("nombre").isNull(), F.lit("No identificado"))
     .otherwise(F.col("nombre"))
)

# Verificación rápida en consola para constatar el cambio
df_consolidado_raw.filter(F.col("nombre") == "No identificado") \
    .select("id_transaccion", "id_cliente", "nombre") \
    .show(5, truncate=False)

 Nombres_Ausentes_en_Clientes  Clientes_No_Registrados_en_Consolidado  Total_Transacciones_Huerfanas
                            0                                       3                             19
+--------------+----------+---------------+
|id_transaccion|id_cliente|nombre         |
+--------------+----------+---------------+
|500398        |7777      |No identificado|
|500317        |7777      |No identificado|
|500288        |9999      |No identificado|
|500247        |8888      |No identificado|
|500191        |7777      |No identificado|
+--------------+----------+---------------+
only showing top 5 rows


### Plazo_dias en ceros

In [40]:
print("==========================================================================")
print("🔍 RADIOGRAFÍA: PLAZOS EN CERO Y NULOS POR TIPO DE PRODUCTO")
print("==========================================================================")

# Agrupamos por tipo de producto y contamos cuántos nulos y cuántos ceros existen
df_analisis_plazo = df_consolidado_raw.groupBy("tipo_producto").agg(
    F.count("*").alias("Total_Registros"),
    F.sum(F.when(F.col("plazo_dias").isNull(), 1).otherwise(0)).alias("Cant_Nulos"),
    F.sum(F.when(F.col("plazo_dias") == 0, 1).otherwise(0)).alias("Cant_Ceros"),
)

df_analisis_plazo.show(truncate=False)

🔍 RADIOGRAFÍA: PLAZOS EN CERO Y NULOS POR TIPO DE PRODUCTO
+----------------+---------------+----------+----------+
|tipo_producto   |Total_Registros|Cant_Nulos|Cant_Ceros|
+----------------+---------------+----------+----------+
|CDT             |144            |56        |0         |
|Credito         |76             |18        |0         |
|Cuenta Ahorros  |143            |92        |51        |
|Cuenta Corriente|60             |45        |15        |
+----------------+---------------+----------+----------+



In [41]:
# Si es Cuenta Ahorros o Cuenta Corriente Y el plazo es 0, lo transformamos en un nulo real
df_consolidado_raw = df_consolidado_raw.withColumn(
    "plazo_dias",
    F.when(
        (F.col("tipo_producto").isin("Cuenta Ahorros", "Cuenta Corriente")) & 
        (F.col("plazo_dias") == 0), 
        F.lit(None)
    ).otherwise(F.col("plazo_dias"))
)

print("==========================================================================")
print("✅ UNIFICACIÓN EXITOSA: Plazos en '0' pasados a null para cuentas.")
print("==========================================================================")

# Verificación rápida: Ya no deberían existir plazos en 0 para estos dos productos
df_consolidado_raw.filter(
    F.col("tipo_producto").isin("Cuenta Ahorros", "Cuenta Corriente")
).groupBy("tipo_producto", "plazo_dias").count().show(truncate=False)

✅ UNIFICACIÓN EXITOSA: Plazos en '0' pasados a null para cuentas.
+----------------+----------+-----+
|tipo_producto   |plazo_dias|count|
+----------------+----------+-----+
|Cuenta Ahorros  |NULL      |143  |
|Cuenta Corriente|NULL      |60   |
+----------------+----------+-----+



### tasa_interes en ceros

In [42]:
print("==========================================================================")
print("🔍 RADIOGRAFÍA: TASAS DE INTERÉS EN CERO Y NULOS POR TIPO DE PRODUCTO")
print("==========================================================================")

# Agrupamos por tipo de producto para auditar el comportamiento de la tasa
df_analisis_tasa = df_consolidado_raw.groupBy("tipo_producto").agg(
    F.count("*").alias("Total_Registros"),
    F.sum(F.when(F.col("tasa_interes").isNull(), 1).otherwise(0)).alias("Cant_Nulos"),
    F.sum(F.when(F.col("tasa_interes") == 0, 1).otherwise(0)).alias("Cant_Ceros"),
)

df_analisis_tasa.show(truncate=False)

🔍 RADIOGRAFÍA: TASAS DE INTERÉS EN CERO Y NULOS POR TIPO DE PRODUCTO
+----------------+---------------+----------+----------+
|tipo_producto   |Total_Registros|Cant_Nulos|Cant_Ceros|
+----------------+---------------+----------+----------+
|CDT             |144            |20        |0         |
|Credito         |76             |12        |0         |
|Cuenta Ahorros  |143            |99        |44        |
|Cuenta Corriente|60             |42        |18        |
+----------------+---------------+----------+----------+



In [43]:
# Si es Cuenta Ahorros o Cuenta Corriente Y la tasa es 0, la transformamos en un nulo real
df_consolidado_raw = df_consolidado_raw.withColumn(
    "tasa_interes",
    F.when(
        (F.col("tipo_producto").isin("Cuenta Ahorros", "Cuenta Corriente")) & 
        (F.col("tasa_interes") == 0), 
        F.lit(None)
    ).otherwise(F.col("tasa_interes"))
)

print("==========================================================================")
print("✅ UNIFICACIÓN EXITOSA: Tasas en '0' pasadas a null para cuentas.")
print("==========================================================================")

# Verificación de control: Ahora el 100% de las cuentas deben figurar con tasa null
df_consolidado_raw.filter(
    F.col("tipo_producto").isin("Cuenta Ahorros", "Cuenta Corriente")
).groupBy("tipo_producto", "tasa_interes").count().show(truncate=False)

✅ UNIFICACIÓN EXITOSA: Tasas en '0' pasadas a null para cuentas.
+----------------+------------+-----+
|tipo_producto   |tasa_interes|count|
+----------------+------------+-----+
|Cuenta Ahorros  |NULL        |143  |
|Cuenta Corriente|NULL        |60   |
+----------------+------------+-----+



### Consistencia en fecha_alta (asociada a cliente)

In [44]:
print("==========================================================================")
print("🔍 DESCRIPTIVO DE fecha_alta (EXCLUYENDO CLIENTES NO IDENTIFICADOS)")
print("==========================================================================")

# Aplicamos el filtro para analizar únicamente a los clientes reales del catálogo
df_descriptivo_fecha = df_consolidado_raw \
    .filter(F.col("nombre") != "No identificado") \
    .agg(
        F.min("fecha_alta").alias("Fecha_Mínima"),
        F.max("fecha_alta").alias("Fecha_Máxima"),
        F.sum(F.when(F.col("fecha_alta").isNull(), 1).otherwise(0)).alias("Clientes_Reales_Sin_Fecha")
    )

df_descriptivo_fecha.show(truncate=False)

🔍 DESCRIPTIVO DE fecha_alta (EXCLUYENDO CLIENTES NO IDENTIFICADOS)
+------------+------------+-------------------------+
|Fecha_Mínima|Fecha_Máxima|Clientes_Reales_Sin_Fecha|
+------------+------------+-------------------------+
|2019-01-11  |2023-12-02  |0                        |
+------------+------------+-------------------------+



### Consistencia en fecha (asociada a transaccion)

In [45]:
print("==========================================================================")
print("🔍 DESCRIPTIVO DE HORIZONTE TEMPORAL: fecha (TRANSACCIONES)")
print("==========================================================================")

# Analizamos los registros en el consolidado para la fecha
df_descriptivo_transacciones = df_consolidado_raw.agg(
    F.min("fecha").alias("Primera_Transaccion_Registrada"),
    F.max("fecha").alias("Ultima_Transaccion_Registrada"),
    F.sum(F.when(F.col("fecha").isNull(), 1).otherwise(0)).alias("Transacciones_Sin_Fecha")
)

df_descriptivo_transacciones.show(truncate=False)

🔍 DESCRIPTIVO DE HORIZONTE TEMPORAL: fecha (TRANSACCIONES)
+------------------------------+-----------------------------+-----------------------+
|Primera_Transaccion_Registrada|Ultima_Transaccion_Registrada|Transacciones_Sin_Fecha|
+------------------------------+-----------------------------+-----------------------+
|2023-02-05                    |2025-12-21                   |0                      |
+------------------------------+-----------------------------+-----------------------+



In [46]:
print("==========================================================================")
print("🔍 AUDITORÍA TEMPORAL: TRANSACCIONES ANTERIORES AL ALTA DEL CLIENTE")
print("==========================================================================")

# Filtramos los registros donde la transacción ocurrió ANTES de la fecha de alta
df_viajes_tiempo = df_consolidado_raw.filter(
    (F.col("nombre") != "No identificado") & 
    (F.col("fecha_alta").isNotNull()) & 
    (F.col("fecha") < F.col("fecha_alta"))
)

total_anomalias = df_viajes_tiempo.count()

print(f"Número de registros con inconsistencia temporal (fecha < fecha_alta): {total_anomalias}")

if total_anomalias > 0:
    print("\n⚠️ MUESTRA DE REGISTROS ANÓMALOS:")
    df_viajes_tiempo.select("id_cliente", "nombre", "fecha_alta", "fecha", "tipo_producto", "monto").show(20, truncate=False)
else:
    print("\n✅ ¡IMPECABLE! No se encontraron transacciones previas a la fecha de alta de los clientes.")

🔍 AUDITORÍA TEMPORAL: TRANSACCIONES ANTERIORES AL ALTA DEL CLIENTE
Número de registros con inconsistencia temporal (fecha < fecha_alta): 1

⚠️ MUESTRA DE REGISTROS ANÓMALOS:
+----------+------------------+----------+----------+----------------+----------+
|id_cliente|nombre            |fecha_alta|fecha     |tipo_producto   |monto     |
+----------+------------------+----------+----------+----------------+----------+
|1001      |María José Álvarez|2023-10-22|2023-03-16|Cuenta Corriente|3749236.00|
+----------+------------------+----------+----------+----------------+----------+



---

## 💾 5. Exportación de Resultados

Se exporta el resultado en un único archivo csv limpio

In [47]:
print("💥 Aplanando 'contacto' y transformando el array 'productos' a string...")

# 2. Extraemos los campos del STRUCT hacia columnas planas y eliminamos la original
df_final_plano = df_consolidado_raw \
    .withColumn("contacto_email", F.col("contacto.email")) \
    .withColumn("contacto_telefono", F.col("contacto.telefono")) \
    .withColumn("productos_str", F.concat_ws(" | ", F.col("productos"))) \
    .drop("contacto", "productos")


💥 Aplanando 'contacto' y transformando el array 'productos' a string...


In [48]:
print("==========================================================================")
print("📐 ESQUEMA FINAL DEL DATASET CONSOLIDADO")
print("==========================================================================")
# Verificamos que los tipos de datos (especialmente Decimal) estén intactos
df_final_plano.printSchema()

print("\n==========================================================================")
print("👀 MUESTRA DE LOS PRIMEROS 5 REGISTROS")
print("==========================================================================")
# Mostramos una muestra limpia sin truncar la información de las columnas
df_final_plano.show(10, truncate=False)

📐 ESQUEMA FINAL DEL DATASET CONSOLIDADO
root
 |-- id_cliente: long (nullable = true)
 |-- id_transaccion: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- tipo_producto: string (nullable = false)
 |-- monto: decimal(18,2) (nullable = true)
 |-- moneda: string (nullable = false)
 |-- tasa_interes: decimal(15,6) (nullable = true)
 |-- plazo_dias: integer (nullable = true)
 |-- canal: string (nullable = false)
 |-- estado: string (nullable = false)
 |-- sucursal: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- activo: boolean (nullable = true)
 |-- contacto_email: string (nullable = true)
 |-- contacto_telefono: string (nullable = true)
 |-- productos_str: string (nullable = false)


👀 MUESTRA DE LOS PRIMEROS 5 REGISTROS
+----------+--------------

In [49]:
# Definimos la ruta de salida (usualmente dentro de la carpeta 'data/processed')
ruta_salida_csv = "../data/processed/consolidado_clientes_transacciones/"

print(f"💾 Guardando dataset consolidado en formato CSV en: {ruta_salida_csv}...")

# Exportamos forzando una única partición, agregando cabeceras y sobreescribiendo si ya existe
df_final_plano.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .csv(ruta_salida_csv)

print("✅ ¡Exportación a CSV completada con éxito!")

💾 Guardando dataset consolidado en formato CSV en: ../data/processed/consolidado_clientes_transacciones/...
✅ ¡Exportación a CSV completada con éxito!
